# Statistical Analysis

**Study 1**: Regression + ANOVA for distance-dependent resistance
**Study 2**: Commitment/persistence comparisons across conditions
**Study 2A/2B**: Framing and attribution effects
**Study 3**: Coalition structure effects

In [3]:
import sys
print(sys.executable)

/Users/wangbaihui/anaconda3/envs/recsys311/bin/python


In [4]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels"])

  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 15.1 MB/s  0:00:00eta 0:00:01
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


0

In [1]:
import json
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import f_oneway, ttest_ind, mannwhitneyu, kruskal, chi2_contingency
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

def get_dist7(probs_dict):
    d = np.array([float(probs_dict.get(str(i), 0) or 0) for i in range(1, 8)])
    return d / d.sum() if d.sum() > 0 else d

def wasserstein_1d(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    if p.sum() > 0: p /= p.sum()
    if q.sum() > 0: q /= q.sum()
    return np.sum(np.abs(np.cumsum(p)[:-1] - np.cumsum(q)[:-1]))

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx-1)*np.std(x,ddof=1)**2 + (ny-1)*np.std(y,ddof=1)**2) / (nx+ny-2))
    return (np.mean(x) - np.mean(y)) / pooled_std if pooled_std > 0 else 0

print("Libraries loaded.")

Libraries loaded.


## 1. Load Data

**Update the paths below to match your local file locations.**

In [20]:
# ══════════════════════════════════════════
# UPDATE THESE PATHS
# ══════════════════════════════════════════
MODELS = {
    "GPT-4o": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/baseline_outputs_explicit_format.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/study1_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/study2_v3_experimental_outputs.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/gpt4o_study2_additional_results.jsonl",
    },
    "DeepSeek-V3.2": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/outputs/deepseek_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/outputs/deepseek_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/deepseek_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/deepseek_study2_additional_results.jsonl",
    },

    "Olmo-3.5": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/olmo_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/olmo_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/olmo_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/olmo_study2_additional_results.jsonl",
    },

    "qwen": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/qwen_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/qwen_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/qwen_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/qwen_study2_additional_results.jsonl",

    },

    'Phi-4': {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/phi4_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/phi4_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/phi4_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/phi4_study2_additional_results.jsonl",

}
}

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def load_baseline(path):
    bl = {}
    for obj in load_jsonl(path):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        md = obj.get("metadata", {})
        qid = md.get("qid") or obj.get("custom_id", "").split("|")[0]
        probs = resp.get("probs", {})
        if qid and probs:
            dist = get_dist7(probs)
            mode = int(np.argmax(dist)) + 1
            bl[qid] = {"dist": dist, "mode": mode, "p_mode": dist[mode-1],
                       "entropy": -np.sum(dist[dist>0] * np.log2(dist[dist>0]))}
    return bl

print("Data loading functions defined.")

Data loading functions defined.


---
# Study 1: Distance-Dependent Resistance

## 1.1 Build Study 1 DataFrame

In [21]:
import json
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import f_oneway, ttest_ind, mannwhitneyu, kruskal, chi2_contingency
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

def get_dist7(probs_dict):
    d = np.array([float(probs_dict.get(str(i), 0) or 0) for i in range(1, 8)])
    return d / d.sum() if d.sum() > 0 else d

def wasserstein_1d(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    if p.sum() > 0: p /= p.sum()
    if q.sum() > 0: q /= q.sum()
    return np.sum(np.abs(np.cumsum(p)[:-1] - np.cumsum(q)[:-1]))

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx-1)*np.std(x,ddof=1)**2 + (ny-1)*np.std(y,ddof=1)**2) / (nx+ny-2))
    return (np.mean(x) - np.mean(y)) / pooled_std if pooled_std > 0 else 0

print("Libraries loaded.")

Libraries loaded.


In [22]:
def build_study1_df(model_name, paths):
    bl = load_baseline(paths["baseline"])
    rows_out = []
    for obj in load_jsonl(paths["study1_full"]):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        probs = resp.get("probs", {})
        if not probs: continue

        md = obj.get("metadata", {})
        qid = md.get("qid")
        if not qid:
            cid = obj.get("custom_id", "")
            qid = cid.split("|")[0]
        if qid not in bl: continue

        post_dist = get_dist7(probs)
        base_info = bl[qid]
        base_dist = base_info["dist"]
        mode = base_info["mode"]
        p_mode_bl = base_info["p_mode"]

        # Get cue info from metadata (works for all models)
        cue_target = md.get("cue_target")
        if cue_target is not None and cue_target != "none":
            cue_target = int(cue_target)
        else:
            cue_target = None

        cue_type = md.get("cue_direction") or md.get("condition", "")
        distance = md.get("cue_distance")
        if distance is not None and distance != "none":
            distance = int(distance)
        else:
            distance = abs(cue_target - mode) if cue_target is not None else None

        # Normalize cue_type names
        cue_type_lower = str(cue_type).lower()
        if "oppose" in cue_type_lower:
            cue_type = "oppose"
        elif "pull" in cue_type_lower:
            cue_type = "pull"
        elif "reinforce" in cue_type_lower or "strong_reinforce" in cue_type_lower:
            cue_type = "reinforce"
        elif "weak_reinforce" in cue_type_lower:
            cue_type = "weak_reinforce"
        elif "memory_only" in cue_type_lower or "baseline" in cue_type_lower or cue_target is None:
            cue_type = "memory_only"
        else:
            cue_type = cue_type_lower

        # Same vs opposite direction
        if cue_target is not None and mode != 4:
            model_side = "left" if mode < 4 else "right"
            cue_side = "left" if cue_target < 4 else ("right" if cue_target > 4 else "neutral")
            same_direction = 1 if model_side == cue_side else 0
        else:
            same_direction = None

        dp_prior = post_dist[mode - 1] - base_dist[mode - 1]
        dp_cue = (post_dist[cue_target - 1] - base_dist[cue_target - 1]) if cue_target else None
        w1 = wasserstein_1d(base_dist, post_dist)

        # Resistance ratio
        resist_ratio = None
        if dp_prior < 0 and dp_cue is not None:
            resist_ratio = 1 - (dp_cue / abs(dp_prior)) if abs(dp_prior) > 0.001 else None

        rows_out.append({
            "model": model_name,
            "qid": qid,
            "mode": mode,
            "p_mode_baseline": p_mode_bl,
            "entropy_baseline": base_info["entropy"],
            "cue_target": cue_target,
            "cue_type": cue_type,
            "distance": distance,
            "same_direction": same_direction,
            "extremity": abs(mode - 4),
            "dp_prior": dp_prior,
            "dp_cue": dp_cue,
            "w1": w1,
            "resist_ratio": resist_ratio,
        })
    return pd.DataFrame(rows_out)

# Rebuild
s1_frames = []
for name, paths in MODELS.items():
    try:
        s1_frames.append(build_study1_df(name, paths))
        print(f"{name}: {len(s1_frames[-1])} trials")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s1 = pd.concat(s1_frames, ignore_index=True)
print(f"\nTotal Study 1 trials: {len(df_s1)}")
print(df_s1.groupby(['model', 'cue_type']).size().unstack(fill_value=0))

GPT-4o: 542 trials
DeepSeek-V3.2: 546 trials
Olmo-3.5: 546 trials
qwen: 546 trials
Phi-4: 546 trials

Total Study 1 trials: 2726
cue_type       memory_only  oppose  pull  reinforce
model                                              
DeepSeek-V3.2           78     260    78        130
GPT-4o                  78     220   132        112
Olmo-3.5                78     312     0        156
Phi-4                   78     268    66        134
qwen                    78     260    78        130


## 1.2 OLS Regression: Distance-Dependent Resistance

DV: ΔP(cue target)
IVs: distance, prior commitment (P(mode)), baseline extremity, cue direction, distance²

In [8]:
print("="*70)
print("STUDY 1 — OLS REGRESSION: ΔP(cue target) ~ Distance + Controls")
print("="*70)

for model_name in df_s1['model'].unique():
    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")

    # Oppose trials only (primary analysis)
    opp = df_s1[(df_s1['model'] == model_name) & (df_s1['cue_type'] == 'oppose')].copy()
    opp = opp.dropna(subset=['dp_cue', 'distance', 'p_mode_baseline'])
    opp['distance_sq'] = opp['distance'] ** 2

    print(f"\nN (oppose trials): {len(opp)}")
    print(f"Distance range: {opp['distance'].min()} – {opp['distance'].max()}")

    # Model 1: Linear distance only
    print("\n--- Model 1: Linear ---")
    m1 = smf.ols('dp_cue ~ distance', data=opp).fit()
    print(f"  R² = {m1.rsquared:.4f}, Adj R² = {m1.rsquared_adj:.4f}")
    print(f"  distance: β = {m1.params['distance']:.4f}, p = {m1.pvalues['distance']:.4e}")

    # Model 2: Quadratic (tests nonlinearity)
    print("\n--- Model 2: Quadratic ---")
    m2 = smf.ols('dp_cue ~ distance + distance_sq', data=opp).fit()
    print(f"  R² = {m2.rsquared:.4f}, Adj R² = {m2.rsquared_adj:.4f}")
    print(f"  distance:    β = {m2.params['distance']:.4f}, p = {m2.pvalues['distance']:.4e}")
    print(f"  distance²:   β = {m2.params['distance_sq']:.4f}, p = {m2.pvalues['distance_sq']:.4e}")

    # Model 3: Full with controls
    print("\n--- Model 3: Full (distance + distance² + controls) ---")
    m3 = smf.ols('dp_cue ~ distance + distance_sq + p_mode_baseline + extremity',
                 data=opp).fit()
    print(f"  R² = {m3.rsquared:.4f}, Adj R² = {m3.rsquared_adj:.4f}")
    print(f"  F({m3.df_model:.0f},{m3.df_resid:.0f}) = {m3.fvalue:.2f}, p = {m3.f_pvalue:.4e}")
    for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
        print(f"  {var:20s}: β = {m3.params[var]:+.4f}, SE = {m3.bse[var]:.4f}, "
              f"t = {m3.tvalues[var]:+.3f}, p = {m3.pvalues[var]:.4e}")

    # Model comparison: linear vs quadratic
    from scipy.stats import f as f_dist
    ssr1 = m1.ssr
    ssr2 = m2.ssr
    df_diff = m2.df_model - m1.df_model
    f_stat = ((ssr1 - ssr2) / df_diff) / (ssr2 / m2.df_resid)
    p_val = 1 - f_dist.cdf(f_stat, df_diff, m2.df_resid)
    print(f"\n  Model comparison (linear vs quadratic):")
    print(f"  F({df_diff},{m2.df_resid:.0f}) = {f_stat:.3f}, p = {p_val:.4e}")
    print(f"  ΔR² = {m2.rsquared - m1.rsquared:.4f}")

    # All cue types regression
    print("\n--- Model 4: All cue types ---")
    all_trials = df_s1[(df_s1['model'] == model_name)].copy()
    all_trials = all_trials.dropna(subset=['dp_cue', 'distance', 'p_mode_baseline'])
    all_trials['distance_sq'] = all_trials['distance'] ** 2
    all_trials = all_trials[all_trials['cue_type'].isin(['oppose', 'pull', 'reinforce'])]

    m4 = smf.ols('dp_cue ~ distance + distance_sq + p_mode_baseline + extremity + C(cue_type)',
                 data=all_trials).fit()
    print(f"  R² = {m4.rsquared:.4f}, Adj R² = {m4.rsquared_adj:.4f}")
    print(f"  F({m4.df_model:.0f},{m4.df_resid:.0f}) = {m4.fvalue:.2f}, p = {m4.f_pvalue:.4e}")
    for var in m4.params.index:
        print(f"  {var:40s}: β = {m4.params[var]:+.4f}, p = {m4.pvalues[var]:.4e}")

STUDY 1 — OLS REGRESSION: ΔP(cue target) ~ Distance + Controls

────────────────────────────────────────────────────────────
Model: GPT-4o
────────────────────────────────────────────────────────────

N (oppose trials): 220
Distance range: 1 – 6

--- Model 1: Linear ---
  R² = 0.0804, Adj R² = 0.0761
  distance: β = -0.0300, p = 1.9680e-05

--- Model 2: Quadratic ---
  R² = 0.1073, Adj R² = 0.0991
  distance:    β = -0.1211, p = 9.7376e-04
  distance²:   β = 0.0116, p = 1.1107e-02

--- Model 3: Full (distance + distance² + controls) ---
  R² = 0.1216, Adj R² = 0.1052
  F(4,215) = 7.44, p = 1.2516e-05
  distance            : β = -0.1332, SE = 0.0390, t = -3.413, p = 7.6767e-04
  distance_sq         : β = +0.0128, SE = 0.0047, t = +2.723, p = 7.0015e-03
  p_mode_baseline     : β = -0.1070, SE = 0.0601, t = -1.781, p = 7.6327e-02
  extremity           : β = +0.0251, SE = 0.0181, t = +1.387, p = 1.6701e-01

  Model comparison (linear vs quadratic):
  F(1.0,217) = 6.560, p = 1.1107e-02
  ΔR

## 1.3 Resistance Ratio: T-test at Threshold (d=2 vs d=3)

In [ ]:
print("="*70)
print("STUDY 1 — MODEL-SPECIFIC THRESHOLD TESTS (directional)")
print("="*70)

# Define candidate thresholds per model based on visual inspection
thresholds = {
    "GPT-4o": [(2, 3), (1, 2), (2, 4)],
    "DeepSeek-V3.2": [(3, 4), (4, 5), (3, 5)],
    "Olmo-3.5": [(1, 3), (2, 4), (3, 5)],
    "qwen": [(1, 2), (1, 3), (2, 4)],
    "Phi-4": [(1, 2), (2, 3), (1, 3)],
}

for model_name in df_s1['model'].unique():
    opp = df_s1[(df_s1['model'] == model_name) & (df_s1['cue_type'] == 'oppose')]
    opp = opp.dropna(subset=['dp_cue'])
    
    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")
    
    # Descriptives by distance
    print("\n  Descriptives (ΔP cue target):")
    for d in sorted(opp['distance'].unique()):
        sub = opp[opp['distance'] == d]['dp_cue']
        print(f"    d={d}: M = {sub.mean():+.4f}, SD = {sub.std():.4f}, n = {len(sub)}")
    
    # Test each candidate threshold (ONE-SIDED: d_low > d_high)
    print(f"\n  Threshold tests (one-sided: ΔP at d_low > ΔP at d_high):")
    for d_low, d_high in thresholds.get(model_name, [(2,3)]):
        g_low = opp[opp['distance'] == d_low]['dp_cue']
        g_high = opp[opp['distance'] == d_high]['dp_cue']
        if len(g_low) > 1 and len(g_high) > 1:
            # One-sided t-test: is d_low mean > d_high mean?
            t, p_two = ttest_ind(g_low, g_high, equal_var=False)
            p_one = p_two / 2 if t > 0 else 1 - p_two / 2  # one-sided
            d_eff = cohens_d(g_low.values, g_high.values)
            sig = '***' if p_one < .001 else ('**' if p_one < .01 else ('*' if p_one < .05 else 'ns'))
            print(f"    d={d_low} vs d={d_high}: "
                  f"M_low={g_low.mean():+.4f} vs M_high={g_high.mean():+.4f}, "
                  f"t={t:.3f}, p(one-sided)={p_one:.4e}, d={d_eff:.3f} {sig}")
    
    # Monotonic trend test: Jonckheere-Terpstra alternative — 
    # test if ΔP(cue) decreases with distance using Spearman correlation
    from scipy.stats import spearmanr
    rho, p_spear = spearmanr(opp['distance'], opp['dp_cue'])
    print(f"\n  Monotonic trend (Spearman): rho = {rho:+.4f}, p = {p_spear:.4e}")
    print(f"    Direction: {'decreasing (resistance increases)' if rho < 0 else 'increasing (accommodation increases)'}")
    
    # Linear contrast: is there a significant negative linear trend?
    from scipy.stats import pearsonr
    r, p_pear = pearsonr(opp['distance'], opp['dp_cue'])
    print(f"  Linear trend (Pearson): r = {r:+.4f}, p = {p_pear:.4e}")

## 1.4 One-Way ANOVA: Resistance Across All Distances

In [ ]:
print("="*70)
print("STUDY 1 — ONE-WAY ANOVA: ΔP(cue) across distances")
print("="*70)

for model_name in df_s1['model'].unique():
    opp = df_s1[(df_s1['model'] == model_name) & (df_s1['cue_type'] == 'oppose')]
    opp = opp.dropna(subset=['dp_cue'])

    groups = [opp[opp['distance'] == d]['dp_cue'].values for d in sorted(opp['distance'].unique())]
    groups = [g for g in groups if len(g) > 1]

    if len(groups) >= 2:
        F, p = f_oneway(*groups)
        n_total = sum(len(g) for g in groups)
        k = len(groups)
        eta_sq = (F * (k - 1)) / (F * (k - 1) + (n_total - k))

        print(f"\n{model_name}:")
        print(f"  F({k-1},{n_total-k}) = {F:.3f}, p = {p:.4e}, η² = {eta_sq:.4f}")

        # Descriptives by distance
        for d in sorted(opp['distance'].unique()):
            sub = opp[opp['distance'] == d]['dp_cue']
            print(f"  d={d}: M = {sub.mean():+.4f}, SD = {sub.std():.4f}, n = {len(sub)}")

        # Tukey HSD post-hoc
        print("\n  Tukey HSD post-hoc:")
        tukey = pairwise_tukeyhsd(opp['dp_cue'], opp['distance'], alpha=0.05)
        print(tukey.summary())

## 1.5 Directional Asymmetry: Oppose vs Reinforce

In [ ]:
print("="*70)
print("STUDY 1 — DIRECTIONAL ASYMMETRY: Oppose vs Reinforce")
print("="*70)

for model_name in df_s1['model'].unique():
    df_m = df_s1[df_s1['model'] == model_name].dropna(subset=['dp_cue'])
    opp = df_m[(df_m['cue_type'] == 'oppose') & (df_m['distance'].isin([1, 2]))]['dp_cue']
    rein = df_m[(df_m['cue_type'] == 'reinforce') & (df_m['distance'].isin([1, 2]))]['dp_cue']

    if len(opp) > 1 and len(rein) > 1:
        t, p = ttest_ind(rein, opp, equal_var=False)
        d_eff = cohens_d(rein.values, opp.values)
        print(f"\n{model_name} (d=1-2):")
        print(f"  Reinforce: M = {rein.mean():+.4f} (SD = {rein.std():.4f}, n = {len(rein)})")
        print(f"  Oppose:    M = {opp.mean():+.4f} (SD = {opp.std():.4f}, n = {len(opp)})")
        print(f"  t = {t:.3f}, p = {p:.4e}, Cohen's d = {d_eff:.3f}")

## 1.6 Neutral vs Committed Susceptibility

In [ ]:
print("="*70)
print("STUDY 1 — NEUTRAL vs COMMITTED SUSCEPTIBILITY")
print("="*70)

for model_name in df_s1['model'].unique():
    df_m = df_s1[df_s1['model'] == model_name].dropna(subset=['w1'])
    pull = df_m[df_m['cue_type'] == 'pull']['w1']
    opp = df_m[df_m['cue_type'] == 'oppose']['w1']

    if len(pull) > 1 and len(opp) > 1:
        # Levene's test for variance
        lev_stat, lev_p = stats.levene(pull, opp)
        t, p = ttest_ind(pull, opp, equal_var=False)
        print(f"\n{model_name}:")
        print(f"  Pull (neutral):    M = {pull.mean():.4f} (SD = {pull.std():.4f}, n = {len(pull)})")
        print(f"  Oppose (committed): M = {opp.mean():.4f} (SD = {opp.std():.4f}, n = {len(opp)})")
        print(f"  Welch's t = {t:.3f}, p = {p:.4e}")
        print(f"  Levene's F = {lev_stat:.3f}, p = {lev_p:.4e} (variance ratio = {pull.var()/opp.var():.2f}x)")

---
# Study 2: Commitment & Correction

In [ ]:
def build_study2_df(model_name, paths):
    bl = load_baseline(paths["baseline"])
    rows = []
    for obj in load_jsonl(paths["study2_additional"]):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        probs = resp.get("probs", {})
        if not probs: continue
        md = obj.get("metadata", {})
        qid = md.get("qid")
        if qid not in bl: continue

        post_dist = get_dist7(probs)
        base_dist = bl[qid]["dist"]
        irr = md["irrational_choice"]
        mode = md["mode_choice"]

        rows.append({
            "model": model_name,
            "qid": qid,
            "experiment": md.get("experiment"),
            "framing": md.get("framing"),
            "attribution": md.get("attribution"),
            "stage": md.get("stage"),
            "irr": irr, "mode": mode,
            "distance": md.get("distance"),
            "w1": wasserstein_1d(base_dist, post_dist),
            "p_irr": post_dist[irr - 1],
            "p_mode": post_dist[mode - 1],
            "committed": int(post_dist[irr - 1] > post_dist[mode - 1]),
            "delta_p_irr": post_dist[irr - 1] - base_dist[irr - 1],
            "p_mode_baseline": bl[qid]["p_mode"],
        })
    return pd.DataFrame(rows)

s2_frames = []
for name, paths in MODELS.items():
    try:
        s2_frames.append(build_study2_df(name, paths))
        print(f"{name}: {len(s2_frames[-1])} trials")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s2 = pd.concat(s2_frames, ignore_index=True)
print(f"\nTotal Study 2 Additional trials: {len(df_s2)}")

## 2.1 Experiment 2A: Chi-square + Logistic Regression for Framing Effect

In [ ]:
print("="*70)
print("EXPERIMENT 2A — FRAMING EFFECT ON COMMITMENT")
print("="*70)

for model_name in df_s2['model'].unique():
    df_m = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == '2A') & (df_s2['stage'] == 'commit')]

    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")

    # Chi-square test on commitment rates
    ct = pd.crosstab(df_m['framing'], df_m['committed'])
    chi2, p, dof, expected = chi2_contingency(ct)
    n = len(df_m)
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"\n  Chi-square test (commitment ~ framing):")
    print(f"  χ²({dof}) = {chi2:.3f}, p = {p:.4e}, Cramér's V = {cramers_v:.3f}")

    # Pairwise chi-square (memory vs suggestion, memory vs instruction, suggestion vs instruction)
    pairs = [('memory', 'suggestion'), ('memory', 'instruction'), ('suggestion', 'instruction')]
    print("\n  Pairwise comparisons (Bonferroni α = 0.017):")
    for f1, f2 in pairs:
        sub = df_m[df_m['framing'].isin([f1, f2])]
        ct2 = pd.crosstab(sub['framing'], sub['committed'])
        chi2_pw, p_pw, _, _ = chi2_contingency(ct2)
        print(f"  {f1} vs {f2}: χ² = {chi2_pw:.3f}, p = {p_pw:.4e} {'***' if p_pw < 0.017 else 'ns'}")

    # Logistic regression: committed ~ framing + distance + p_mode_baseline
    print("\n  Logistic regression (committed ~ framing + controls):")
    df_m_copy = df_m.copy()
    df_m_copy['framing_code'] = df_m_copy['framing'].map({'memory': 2, 'instruction': 1, 'suggestion': 0})
    try:
        logit = smf.logit('committed ~ framing_code + distance + p_mode_baseline', data=df_m_copy).fit(disp=0)
        print(f"  Pseudo R² = {logit.prsquared:.4f}")
        for var in ['framing_code', 'distance', 'p_mode_baseline']:
            or_val = np.exp(logit.params[var])
            print(f"  {var:20s}: β = {logit.params[var]:+.4f}, OR = {or_val:.3f}, "
                  f"z = {logit.tvalues[var]:+.3f}, p = {logit.pvalues[var]:.4e}")
    except Exception as e:
        print(f"  Logistic regression failed: {e}")

    # ANOVA on W₁
    print("\n  One-way ANOVA on W₁ (distributional shift):")
    groups_w1 = [df_m[df_m['framing'] == f]['w1'].values for f in ['memory', 'suggestion', 'instruction']]
    F, p = f_oneway(*groups_w1)
    print(f"  F(2,{len(df_m)-3}) = {F:.3f}, p = {p:.4e}")
    tukey_w1 = pairwise_tukeyhsd(df_m['w1'], df_m['framing'], alpha=0.05)
    print(tukey_w1.summary())

## 2.2 Experiment 2A: Persistence Comparison Across Framings

In [ ]:
print("="*70)
print("EXPERIMENT 2A — PERSISTENCE COMPARISON")
print("="*70)

for model_name in df_s2['model'].unique():
    df_m = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == '2A')]
    commit_df = df_m[df_m['stage'] == 'commit']
    correct_df = df_m[df_m['stage'] == 'correct']

    print(f"\n--- {model_name} ---")

    persist_data = []
    for f in ['memory', 'suggestion', 'instruction']:
        committed = commit_df[(commit_df['framing'] == f) & (commit_df['committed'] == 1)]
        keys = set(zip(committed['qid'], committed['irr']))
        corrections = correct_df[
            (correct_df['framing'] == f) &
            correct_df.apply(lambda x: (x['qid'], x['irr']) in keys, axis=1)
        ]
        for _, row in corrections.iterrows():
            persist_data.append({'framing': f, 'persisted': row['committed']})

    pdf = pd.DataFrame(persist_data)
    if len(pdf) > 0:
        # Chi-square on persistence
        ct = pd.crosstab(pdf['framing'], pdf['persisted'])
        if ct.shape[1] >= 2:
            chi2, p, dof, _ = chi2_contingency(ct)
            print(f"  Chi-square (persistence ~ framing): χ²({dof}) = {chi2:.3f}, p = {p:.4f}")

        for f in ['memory', 'suggestion', 'instruction']:
            sub = pdf[pdf['framing'] == f]
            rate = sub['persisted'].mean() * 100 if len(sub) > 0 else 0
            print(f"  {f}: {rate:.1f}% persist (n={len(sub)})")

## 2.3 Experiment 2B: Attribution Effect

In [ ]:
print("="*70)
print("EXPERIMENT 2B — ATTRIBUTION EFFECT ON COMMITMENT")
print("="*70)

for model_name in df_s2['model'].unique():
    df_m = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == '2B') & (df_s2['stage'] == 'commit')]

    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")

    # Chi-square
    ct = pd.crosstab(df_m['attribution'], df_m['committed'])
    chi2, p, dof, _ = chi2_contingency(ct)
    n = len(df_m)
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"\n  Chi-square (commitment ~ attribution):")
    print(f"  χ²({dof}) = {chi2:.3f}, p = {p:.4e}, Cramér's V = {cramers_v:.3f}")

    # Critical test: Self vs User (sycophancy predicts User > Self)
    self_commit = df_m[df_m['attribution'] == 'self']['committed']
    user_commit = df_m[df_m['attribution'] == 'user']['committed']
    other_commit = df_m[df_m['attribution'] == 'other_ai']['committed']

    print("\n  Critical pairwise comparisons:")
    for name1, g1, name2, g2 in [
        ('self', self_commit, 'user', user_commit),
        ('self', self_commit, 'other_ai', other_commit),
        ('user', user_commit, 'other_ai', other_commit),
    ]:
        ct2 = pd.DataFrame({'attr': [name1]*len(g1) + [name2]*len(g2),
                           'committed': list(g1) + list(g2)})
        ct2_tab = pd.crosstab(ct2['attr'], ct2['committed'])
        if ct2_tab.shape[1] >= 2:
            chi2_pw, p_pw, _, _ = chi2_contingency(ct2_tab)
            print(f"  {name1} vs {name2}: χ² = {chi2_pw:.3f}, p = {p_pw:.4e} "
                  f"({name1}: {g1.mean()*100:.1f}% vs {name2}: {g2.mean()*100:.1f}%)")

    # Logistic regression
    print("\n  Logistic regression (committed ~ attribution + controls):")
    df_m_copy = df_m.copy()
    df_m_copy['attr_code'] = df_m_copy['attribution'].map({'self': 2, 'user': 1, 'other_ai': 0})
    try:
        logit = smf.logit('committed ~ attr_code + distance + p_mode_baseline', data=df_m_copy).fit(disp=0)
        print(f"  Pseudo R² = {logit.prsquared:.4f}")
        for var in ['attr_code', 'distance', 'p_mode_baseline']:
            or_val = np.exp(logit.params[var])
            print(f"  {var:20s}: β = {logit.params[var]:+.4f}, OR = {or_val:.3f}, "
                  f"z = {logit.tvalues[var]:+.3f}, p = {logit.pvalues[var]:.4e}")
    except Exception as e:
        print(f"  Logistic regression failed: {e}")

    # ANOVA on W₁
    print("\n  One-way ANOVA on W₁:")
    groups_w1 = [df_m[df_m['attribution'] == a]['w1'].values for a in ['self', 'other_ai', 'user']]
    F, p = f_oneway(*groups_w1)
    print(f"  F(2,{len(df_m)-3}) = {F:.3f}, p = {p:.4e}")
    tukey_w1 = pairwise_tukeyhsd(df_m['w1'], df_m['attribution'], alpha=0.05)
    print(tukey_w1.summary())

## 2.4 Experiment 2B: Persistence by Attribution

In [ ]:
print("="*70)
print("EXPERIMENT 2B — ATTRIBUTION EFFECT ON COMMITMENT")
print("="*70)

for model_name in df_s2['model'].unique():
    df_m = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == '2B') & (df_s2['stage'] == 'commit')]

    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")

    # Descriptives
    for a in ['self', 'other_ai', 'user']:
        sub = df_m[df_m['attribution'] == a]
        print(f"  {a}: commit = {sub['committed'].mean()*100:.1f}% (n={len(sub)}), "
              f"W₁ = {sub['w1'].mean():.3f}, ΔP(irr) = {sub['delta_p_irr'].mean():+.3f}")

    # Chi-square omnibus
    ct = pd.crosstab(df_m['attribution'], df_m['committed'])
    chi2, p, dof, _ = chi2_contingency(ct)
    n = len(df_m)
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"\n  Chi-square (commitment ~ attribution):")
    print(f"  χ²({dof}) = {chi2:.3f}, p = {p:.4e}, Cramér's V = {cramers_v:.3f}")

    # Critical pairwise: Self vs User, Self vs Other-AI, User vs Other-AI
    print("\n  Pairwise comparisons (Bonferroni α = 0.017):")
    for a1, a2 in [('self', 'user'), ('self', 'other_ai'), ('user', 'other_ai')]:
        g1 = df_m[df_m['attribution'] == a1]['committed']
        g2 = df_m[df_m['attribution'] == a2]['committed']
        sub = df_m[df_m['attribution'].isin([a1, a2])]
        ct2 = pd.crosstab(sub['attribution'], sub['committed'])
        if ct2.shape[1] >= 2:
            chi2_pw, p_pw, _, _ = chi2_contingency(ct2)
            sig = '***' if p_pw < 0.017 else ('*' if p_pw < 0.05 else 'ns')
            print(f"  {a1} ({g1.mean()*100:.1f}%) vs {a2} ({g2.mean()*100:.1f}%): "
                  f"χ² = {chi2_pw:.3f}, p = {p_pw:.4e} {sig}")
        else:
            print(f"  {a1} vs {a2}: cannot compute (no variation)")

    # Logistic regression
    print("\n  Logistic regression (committed ~ attribution + controls):")
    df_m_copy = df_m.copy()
    df_m_copy['attr_code'] = df_m_copy['attribution'].map({'self': 2, 'user': 1, 'other_ai': 0})
    try:
        logit = smf.logit('committed ~ attr_code + distance + p_mode_baseline', data=df_m_copy).fit(disp=0)
        print(f"  Pseudo R² = {logit.prsquared:.4f}")
        for var in ['attr_code', 'distance', 'p_mode_baseline']:
            or_val = np.exp(logit.params[var])
            print(f"  {var:20s}: β = {logit.params[var]:+.4f}, OR = {or_val:.3f}, "
                  f"z = {logit.tvalues[var]:+.3f}, p = {logit.pvalues[var]:.4e}")
    except Exception as e:
        print(f"  Logistic regression failed: {e}")

    # ANOVA on W₁
    print("\n  One-way ANOVA on W₁:")
    groups_w1 = [df_m[df_m['attribution'] == a]['w1'].values for a in ['self', 'other_ai', 'user']]
    F, p = f_oneway(*groups_w1)
    print(f"  F(2,{len(df_m)-3}) = {F:.3f}, p = {p:.4e}")
    tukey = pairwise_tukeyhsd(df_m['w1'], df_m['attribution'], alpha=0.05)
    print(tukey.summary())

In [ ]:
print("="*70)
print("STUDY 1 — ADDITIONAL REGRESSIONS: W₁ and Commitment Rate")
print("="*70)

for model_name in df_s1['model'].unique():
    opp = df_s1[(df_s1['model'] == model_name) & (df_s1['cue_type'] == 'oppose')].copy()
    opp = opp.dropna(subset=['w1', 'distance', 'p_mode_baseline'])
    opp['distance_sq'] = opp['distance'] ** 2
    
    # Define "committed to cue" as dp_cue > 0.10
    opp['moved_to_cue'] = (opp['dp_cue'] > 0.10).astype(int)
    
    print(f"\n{'─'*60}")
    print(f"Model: {model_name} (N={len(opp)} oppose trials)")
    print(f"{'─'*60}")
    
    # ── DV: W₁ (total distributional movement) ──
    print("\n  --- DV: W₁ (OLS) ---")
    try:
        m_w1 = smf.ols('w1 ~ distance + distance_sq + p_mode_baseline + extremity', data=opp).fit()
        print(f"  R² = {m_w1.rsquared:.4f}, F({m_w1.df_model:.0f},{m_w1.df_resid:.0f}) = {m_w1.fvalue:.2f}, p = {m_w1.f_pvalue:.4e}")
        for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
            print(f"    {var:20s}: β = {m_w1.params[var]:+.4f}, p = {m_w1.pvalues[var]:.4e}")
    except Exception as e:
        print(f"  Failed: {e}")
    
    # ── DV: Commitment rate (Logistic) ──
    print("\n  --- DV: Moved to cue (Logistic) ---")
    print(f"  Base rate: {opp['moved_to_cue'].mean()*100:.1f}%")
    try:
        m_logit = smf.logit('moved_to_cue ~ distance + distance_sq + p_mode_baseline + extremity', data=opp).fit(disp=0)
        print(f"  Pseudo R² = {m_logit.prsquared:.4f}")
        for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
            or_val = np.exp(m_logit.params[var])
            print(f"    {var:20s}: β = {m_logit.params[var]:+.4f}, OR = {or_val:.3f}, p = {m_logit.pvalues[var]:.4e}")
    except Exception as e:
        print(f"  Failed: {e}")

print("\n\n" + "="*70)
print("STUDY 2 — REGRESSIONS: W₁ and Commitment Rate by Framing/Attribution")
print("="*70)

for model_name in df_s2['model'].unique():
    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")
    
    # ── Exp 2A: W₁ ~ framing + controls ──
    df_2a = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == '2A') & (df_s2['stage'] == 'commit')].copy()
    df_2a['framing_code'] = df_2a['framing'].map({'memory': 2, 'instruction': 1, 'suggestion': 0})
    
    if len(df_2a) > 10:
        print("\n  --- Exp 2A: DV = W₁ (OLS) ---")
        try:
            m = smf.ols('w1 ~ framing_code + distance + p_mode_baseline', data=df_2a).fit()
            print(f"  R² = {m.rsquared:.4f}, F = {m.fvalue:.2f}, p = {m.f_pvalue:.4e}")
            for var in ['framing_code', 'distance', 'p_mode_baseline']:
                print(f"    {var:20s}: β = {m.params[var]:+.4f}, p = {m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"  Failed: {e}")

        print("\n  --- Exp 2A: DV = committed (Logistic) ---")
        try:
            m = smf.logit('committed ~ framing_code + distance + p_mode_baseline', data=df_2a).fit(disp=0)
            print(f"  Pseudo R² = {m.prsquared:.4f}")
            for var in ['framing_code', 'distance', 'p_mode_baseline']:
                or_val = np.exp(m.params[var])
                print(f"    {var:20s}: β = {m.params[var]:+.4f}, OR = {or_val:.3f}, p = {m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"  Failed: {e}")
    
    # ── Exp 2B: W₁ ~ attribution + controls ──
    df_2b = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == '2B') & (df_s2['stage'] == 'commit')].copy()
    df_2b['attr_code'] = df_2b['attribution'].map({'self': 2, 'user': 1, 'other_ai': 0})
    
    if len(df_2b) > 10:
        print("\n  --- Exp 2B: DV = W₁ (OLS) ---")
        try:
            m = smf.ols('w1 ~ attr_code + distance + p_mode_baseline', data=df_2b).fit()
            print(f"  R² = {m.rsquared:.4f}, F = {m.fvalue:.2f}, p = {m.f_pvalue:.4e}")
            for var in ['attr_code', 'distance', 'p_mode_baseline']:
                print(f"    {var:20s}: β = {m.params[var]:+.4f}, p = {m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"  Failed: {e}")

        print("\n  --- Exp 2B: DV = committed (Logistic) ---")
        try:
            m = smf.logit('committed ~ attr_code + distance + p_mode_baseline', data=df_2b).fit(disp=0)
            print(f"  Pseudo R² = {m.prsquared:.4f}")
            for var in ['attr_code', 'distance', 'p_mode_baseline']:
                or_val = np.exp(m.params[var])
                print(f"    {var:20s}: β = {m.params[var]:+.4f}, OR = {or_val:.3f}, p = {m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"  Failed: {e}")

In [21]:
print("="*70)
print("STUDY 1 — ADDITIONAL REGRESSIONS: W₁ and Commitment Rate")
print("="*70)

for model_name in df_s1['model'].unique():
    opp = df_s1[(df_s1['model'] == model_name) & (df_s1['cue_type'] == 'oppose')].copy()
    opp = opp.dropna(subset=['w1', 'distance', 'p_mode_baseline'])
    opp['distance_sq'] = opp['distance'] ** 2
    
    # Define "committed to cue" as dp_cue > 0.10
    opp['moved_to_cue'] = (opp['dp_cue'] > 0.10).astype(int)
    
    print(f"\n{'─'*60}")
    print(f"Model: {model_name} (N={len(opp)} oppose trials)")
    print(f"{'─'*60}")
    
    # ── DV: W₁ (total distributional movement) ──
    print("\n  --- DV: W₁ (OLS) ---")
    try:
        m_w1 = smf.ols('w1 ~ distance + distance_sq + p_mode_baseline + extremity', data=opp).fit()
        print(f"  R² = {m_w1.rsquared:.4f}, F({m_w1.df_model:.0f},{m_w1.df_resid:.0f}) = {m_w1.fvalue:.2f}, p = {m_w1.f_pvalue:.4e}")
        for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
            print(f"    {var:20s}: β = {m_w1.params[var]:+.4f}, p = {m_w1.pvalues[var]:.4e}")
    except Exception as e:
        print(f"  Failed: {e}")
    
    # ── DV: Commitment rate (Logistic) ──
    print("\n  --- DV: Moved to cue (Logistic) ---")
    print(f"  Base rate: {opp['moved_to_cue'].mean()*100:.1f}%")
    try:
        m_logit = smf.logit('moved_to_cue ~ distance + distance_sq + p_mode_baseline + extremity', data=opp).fit(disp=0)
        print(f"  Pseudo R² = {m_logit.prsquared:.4f}")
        for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
            or_val = np.exp(m_logit.params[var])
            print(f"    {var:20s}: β = {m_logit.params[var]:+.4f}, OR = {or_val:.3f}, p = {m_logit.pvalues[var]:.4e}")
    except Exception as e:
        print(f"  Failed: {e}")

print("\n\n" + "="*70)
print("STUDY 2 — REGRESSIONS: W₁ and Commitment Rate by Framing/Attribution")
print("="*70)

for model_name in df_s2['model'].unique():
    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")
    
    # ── Exp 2A: W₁ ~ framing + controls ──
    df_2a = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == '2A') & (df_s2['stage'] == 'commit')].copy()
    df_2a['framing_code'] = df_2a['framing'].map({'memory': 2, 'instruction': 1, 'suggestion': 0})
    
    if len(df_2a) > 10:
        print("\n  --- Exp 2A: DV = W₁ (OLS) ---")
        try:
            m = smf.ols('w1 ~ framing_code + distance + p_mode_baseline', data=df_2a).fit()
            print(f"  R² = {m.rsquared:.4f}, F = {m.fvalue:.2f}, p = {m.f_pvalue:.4e}")
            for var in ['framing_code', 'distance', 'p_mode_baseline']:
                print(f"    {var:20s}: β = {m.params[var]:+.4f}, p = {m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"  Failed: {e}")

        print("\n  --- Exp 2A: DV = committed (Logistic) ---")
        try:
            m = smf.logit('committed ~ framing_code + distance + p_mode_baseline', data=df_2a).fit(disp=0)
            print(f"  Pseudo R² = {m.prsquared:.4f}")
            for var in ['framing_code', 'distance', 'p_mode_baseline']:
                or_val = np.exp(m.params[var])
                print(f"    {var:20s}: β = {m.params[var]:+.4f}, OR = {or_val:.3f}, p = {m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"  Failed: {e}")
    
    # ── Exp 2B: W₁ ~ attribution + controls ──
    df_2b = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == '2B') & (df_s2['stage'] == 'commit')].copy()
    df_2b['attr_code'] = df_2b['attribution'].map({'self': 2, 'user': 1, 'other_ai': 0})
    
    if len(df_2b) > 10:
        print("\n  --- Exp 2B: DV = W₁ (OLS) ---")
        try:
            m = smf.ols('w1 ~ attr_code + distance + p_mode_baseline', data=df_2b).fit()
            print(f"  R² = {m.rsquared:.4f}, F = {m.fvalue:.2f}, p = {m.f_pvalue:.4e}")
            for var in ['attr_code', 'distance', 'p_mode_baseline']:
                print(f"    {var:20s}: β = {m.params[var]:+.4f}, p = {m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"  Failed: {e}")

        print("\n  --- Exp 2B: DV = committed (Logistic) ---")
        try:
            m = smf.logit('committed ~ attr_code + distance + p_mode_baseline', data=df_2b).fit(disp=0)
            print(f"  Pseudo R² = {m.prsquared:.4f}")
            for var in ['attr_code', 'distance', 'p_mode_baseline']:
                or_val = np.exp(m.params[var])
                print(f"    {var:20s}: β = {m.params[var]:+.4f}, OR = {or_val:.3f}, p = {m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"  Failed: {e}")

STUDY 1 — ADDITIONAL REGRESSIONS: W₁ and Commitment Rate

────────────────────────────────────────────────────────────
Model: GPT-4o (N=220 oppose trials)
────────────────────────────────────────────────────────────

  --- DV: W₁ (OLS) ---
  R² = 0.1511, F(4,215) = 9.56, p = 3.9021e-07
    distance            : β = +0.1478, p = 2.3327e-01
    distance_sq         : β = -0.0187, p = 2.1012e-01
    p_mode_baseline     : β = -0.6509, p = 7.4802e-04
    extremity           : β = -0.1131, p = 5.0005e-02

  --- DV: Moved to cue (Logistic) ---
  Base rate: 5.0%
  Pseudo R² = 0.2549
    distance            : β = -1.7832, OR = 0.168, p = 2.6033e-01
    distance_sq         : β = +0.0573, OR = 1.059, p = 8.1375e-01
    p_mode_baseline     : β = -2.5083, OR = 0.081, p = 1.8170e-01
    extremity           : β = +0.7954, OR = 2.215, p = 2.7704e-01

────────────────────────────────────────────────────────────
Model: DeepSeek-V3.2 (N=260 oppose trials)
──────────────────────────────────────────────────

In [23]:
# Check Study 2 original data structure
import json

with open("study2_v3_experimental_outputs.jsonl") as f:
    rows = [json.loads(l) for l in f if l.strip()]

print(f"Total rows: {len(rows)}")
print(f"\nSample metadata keys:")
for r in rows[:3]:
    md = r.get('metadata', {})
    print(f"  {list(md.keys())}")
    print(f"  condition: {md.get('condition')}, stage: {md.get('stage')}")

# Count by stage/condition
from collections import Counter
stages = Counter()
for r in rows:
    md = r.get('metadata', {})
    stages[(md.get('condition'), md.get('stage'))] += 1
print(f"\nStage counts:")
for k, v in sorted(stages.items()):
    print(f"  {k}: {v}")

Total rows: 435

Sample metadata keys:
  ['phase', 'study', 'qid', 'irrational_choice', 'mode_choice', 'mode_prob', 'irrational_prob', 'distance', 'condition', 'has_justification', 'has_correction']
  condition: memory_only, stage: None
  ['phase', 'study', 'qid', 'irrational_choice', 'mode_choice', 'mode_prob', 'irrational_prob', 'distance', 'condition', 'has_justification', 'has_correction']
  condition: memory_only, stage: None
  ['phase', 'study', 'qid', 'irrational_choice', 'mode_choice', 'mode_prob', 'irrational_prob', 'distance', 'condition', 'has_justification', 'has_correction']
  condition: memory_only, stage: None

Stage counts:
  ('memory_justify', None): 145
  ('memory_justify_correction', None): 145
  ('memory_only', None): 145


In [24]:
with open("study2_v3_experimental_outputs.jsonl") as f:
    rows = [json.loads(l) for l in f if l.strip()]

print(f"Total rows: {len(rows)}")

# Check a few samples
for r in rows[:5]:
    md = r.get('metadata', {})
    resp = r.get('response', {})
    print(f"  custom_id: {r.get('custom_id','')[:50]}")
    print(f"  metadata keys: {list(md.keys())}")
    print(f"  response keys: {list(resp.keys())}")
    print()

Total rows: 435
  custom_id: moral_5|memory_only|irr5
  metadata keys: ['phase', 'study', 'qid', 'irrational_choice', 'mode_choice', 'mode_prob', 'irrational_prob', 'distance', 'condition', 'has_justification', 'has_correction']
  response keys: ['status_code', 'choice', 'justification', 'raw_content', 'logprobs', 'probs', 'mean', 'entropy_norm']

  custom_id: moral_5|memory_only|irr6
  metadata keys: ['phase', 'study', 'qid', 'irrational_choice', 'mode_choice', 'mode_prob', 'irrational_prob', 'distance', 'condition', 'has_justification', 'has_correction']
  response keys: ['status_code', 'choice', 'justification', 'raw_content', 'logprobs', 'probs', 'mean', 'entropy_norm']

  custom_id: moral_4|memory_only|irr2
  metadata keys: ['phase', 'study', 'qid', 'irrational_choice', 'mode_choice', 'mode_prob', 'irrational_prob', 'distance', 'condition', 'has_justification', 'has_correction']
  response keys: ['status_code', 'choice', 'justification', 'raw_content', 'logprobs', 'probs', 'mean',

In [26]:
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import ttest_ind

def get_dist7(probs_dict):
    d = np.array([float(probs_dict.get(str(i), 0) or 0) for i in range(1, 8)])
    return d / d.sum() if d.sum() > 0 else d

def wasserstein_1d(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    if p.sum() > 0: p /= p.sum()
    if q.sum() > 0: q /= q.sum()
    return np.sum(np.abs(np.cumsum(p)[:-1] - np.cumsum(q)[:-1]))

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    s = np.sqrt(((nx-1)*np.std(x,ddof=1)**2 + (ny-1)*np.std(y,ddof=1)**2) / (nx+ny-2))
    return (np.mean(x) - np.mean(y)) / s if s > 0 else 0

# ══════════════════════════════════════════════════════════
# UPDATE THESE PATHS
# ══════════════════════════════════════════════════════════
STUDY2_FILES = {
    "GPT-4o": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/study2_v3_experimental_outputs.jsonl",
    "DeepSeek-V3.2": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/deepseek_study2_additional_results.jsonl",
    "Qwen-2.5": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/qwen_study2_additional_results.jsonl",
    "Phi-4": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/phi4_study2_additional_results.jsonl",
}

def build_study2_orig_df(path, model_name):
    with open(path) as f:
        rows = [json.loads(l) for l in f if l.strip()]

    records = {}
    for r in rows:
        resp = r.get('response', {})
        if resp.get('status_code') != 200: continue
        md = r.get('metadata', {})

        qid = md.get('qid')
        irr = md.get('irrational_choice')
        condition = md.get('condition', '')
        if not qid or irr is None: continue

        probs = resp.get('probs', {})
        if not probs: continue
        dist = get_dist7(probs)

        key = (qid, irr)
        if key not in records:
            records[key] = {
                'qid': qid, 'irr': irr,
                'mode': md.get('mode_choice'),
                'distance': md.get('distance'),
                'p_mode_baseline': md.get('mode_prob', md.get('p_mode_baseline')),
                'p_irr_baseline': md.get('irrational_prob', md.get('p_irr_baseline')),
            }

        records[key][f'{condition}_dist'] = dist
        records[key][f'{condition}_p_irr'] = dist[irr - 1]
        mode = records[key]['mode']
        if mode is not None:
            records[key][f'{condition}_p_mode'] = dist[int(mode) - 1]

    rows_out = []
    for key, rec in records.items():
        irr = rec['irr']
        mode = rec['mode']
        if mode is None: continue
        mode = int(mode)
        p_irr_bl = float(rec.get('p_irr_baseline', 0) or 0)
        p_mode_bl = float(rec.get('p_mode_baseline', 0) or 0)

        # D1: commitment stage — try memory_justify first, then memory_only
        d1_dist = None
        for cond in ['memory_justify', 'memory_only']:
            if f'{cond}_dist' in rec:
                d1_dist = rec[f'{cond}_dist']
                break
        if d1_dist is None: continue

        d1_p_irr = d1_dist[irr - 1]
        d1_p_mode = d1_dist[mode - 1]
        committed = d1_p_irr > d1_p_mode

        # Construct baseline dist from metadata probabilities
        # (approximate — used only for W1 calculation)
        bl_dist = np.zeros(7)
        bl_dist[mode - 1] = p_mode_bl
        bl_dist[irr - 1] = p_irr_bl
        remaining = max(0, 1.0 - p_mode_bl - p_irr_bl)
        for j in range(7):
            if j != mode - 1 and j != irr - 1:
                bl_dist[j] = remaining / 5.0
        if bl_dist.sum() > 0:
            bl_dist /= bl_dist.sum()

        row = {
            'model': model_name,
            'qid': rec['qid'],
            'irr': irr,
            'mode': mode,
            'distance': rec.get('distance'),
            'p_mode_baseline': p_mode_bl,
            'p_irr_baseline': p_irr_bl,
            'committed_d1': int(committed),
            'd1_p_irr': d1_p_irr,
            'd1_p_mode': d1_p_mode,
            # Correct DVs: shift from baseline
            'delta_p_irr_d1': d1_p_irr - p_irr_bl,       # how much P(irr) increased
            'delta_p_mode_d1': d1_p_mode - p_mode_bl,     # how much P(mode) decreased
            'w1_bl_d1': wasserstein_1d(bl_dist, d1_dist),  # total movement at D1
            # Relative standing (used as predictor for persistence)
            'd1_commitment_strength': d1_p_irr - d1_p_mode,
        }

        # D2: correction stage
        if 'memory_justify_correction_dist' in rec:
            d2_dist = rec['memory_justify_correction_dist']
            d2_p_irr = d2_dist[irr - 1]
            d2_p_mode = d2_dist[mode - 1]
            persisted = d2_p_irr > d2_p_mode

            row['persisted_d2'] = int(persisted)
            row['d2_p_irr'] = d2_p_irr
            row['d2_p_mode'] = d2_p_mode
            row['delta_p_irr_d2'] = d2_p_irr - p_irr_bl      # residual shift from baseline
            row['delta_p_mode_d2'] = d2_p_mode - p_mode_bl    # residual mode displacement
            row['w1_d1_d2'] = wasserstein_1d(d1_dist, d2_dist) # correction magnitude
            row['w1_bl_d2'] = wasserstein_1d(bl_dist, d2_dist)  # residual total displacement

        rows_out.append(row)

    return pd.DataFrame(rows_out)

# ══════════════════════════════════════════════════════════
# Load all models
# ══════════════════════════════════════════════════════════
all_frames = []
for model_name, path in STUDY2_FILES.items():
    try:
        df = build_study2_orig_df(path, model_name)
        all_frames.append(df)
        print(f"{model_name}: {len(df)} trials, "
              f"committed={df['committed_d1'].sum()} ({df['committed_d1'].mean()*100:.1f}%)")
    except Exception as e:
        print(f"{model_name}: FAILED — {e}")

df_all = pd.concat(all_frames, ignore_index=True)
print(f"\nTotal: {len(df_all)} trials across {df_all['model'].nunique()} models")

# ══════════════════════════════════════════════════════════
# Run regressions for each model
# ══════════════════════════════════════════════════════════

for model_name in df_all['model'].unique():
    df_m = df_all[df_all['model'] == model_name].copy()
    df_m['distance'] = pd.to_numeric(df_m['distance'], errors='coerce')
    df_m['p_mode_baseline'] = pd.to_numeric(df_m['p_mode_baseline'], errors='coerce')
    df_m['p_irr_baseline'] = pd.to_numeric(df_m['p_irr_baseline'], errors='coerce')
    df_m = df_m.dropna(subset=['distance', 'p_mode_baseline'])

    print(f"\n{'═'*70}")
    print(f"  {model_name} (N={len(df_m)})")
    print(f"{'═'*70}")

    # ── Descriptives ──
    print(f"\n  Descriptives:")
    print(f"    Commitment rate: {df_m['committed_d1'].mean()*100:.1f}%")
    print(f"    Mean ΔP(irr) at D1: {df_m['delta_p_irr_d1'].mean():+.3f}")
    print(f"    Mean W₁(baseline→D1): {df_m['w1_bl_d1'].mean():.3f}")
    if 'persisted_d2' in df_m.columns:
        comm = df_m[df_m['committed_d1'] == 1]
        if len(comm) > 0 and 'persisted_d2' in comm.columns:
            print(f"    Persistence (of committed): {comm['persisted_d2'].mean()*100:.1f}% "
                  f"({int(comm['persisted_d2'].sum())}/{len(comm)})")
        if 'delta_p_irr_d2' in df_m.columns:
            print(f"    Mean residual ΔP(irr) at D2: {df_m['delta_p_irr_d2'].mean():+.3f}")

    # ══════════════════════════════════════════════════════
    # REGRESSION 1a: DV = committed (binary, Logistic)
    # ══════════════════════════════════════════════════════
    print(f"\n  {'─'*60}")
    print(f"  REG 1a: DV = committed_d1 (Logistic)")
    print(f"  {'─'*60}")
    try:
        m = smf.logit('committed_d1 ~ distance + p_mode_baseline + p_irr_baseline',
                       data=df_m).fit(disp=0)
        print(f"  Pseudo R² = {m.prsquared:.4f}, N = {int(m.nobs)}")
        for var in ['distance', 'p_mode_baseline', 'p_irr_baseline']:
            if var in m.params:
                or_val = np.exp(m.params[var])
                print(f"    {var:25s}: β={m.params[var]:+.4f}, OR={or_val:.3f}, "
                      f"z={m.tvalues[var]:+.3f}, p={m.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Failed: {e}")

    # ══════════════════════════════════════════════════════
    # REGRESSION 1b: DV = ΔP(irr) at D1 (continuous, OLS)
    # ══════════════════════════════════════════════════════
    print(f"\n  {'─'*60}")
    print(f"  REG 1b: DV = delta_p_irr_d1 (OLS)")
    print(f"  {'─'*60}")
    try:
        m = smf.ols('delta_p_irr_d1 ~ distance + p_mode_baseline + p_irr_baseline',
                      data=df_m).fit()
        print(f"  R² = {m.rsquared:.4f}, F({m.df_model:.0f},{m.df_resid:.0f})={m.fvalue:.2f}, "
              f"p={m.f_pvalue:.4e}")
        for var in ['distance', 'p_mode_baseline', 'p_irr_baseline']:
            if var in m.params:
                print(f"    {var:25s}: β={m.params[var]:+.4f}, p={m.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Failed: {e}")

    # ══════════════════════════════════════════════════════
    # REGRESSION 1c: DV = W₁(baseline→D1) (continuous, OLS)
    # ══════════════════════════════════════════════════════
    print(f"\n  {'─'*60}")
    print(f"  REG 1c: DV = w1_bl_d1 (OLS)")
    print(f"  {'─'*60}")
    try:
        m = smf.ols('w1_bl_d1 ~ distance + p_mode_baseline + p_irr_baseline',
                      data=df_m).fit()
        print(f"  R² = {m.rsquared:.4f}, F({m.df_model:.0f},{m.df_resid:.0f})={m.fvalue:.2f}, "
              f"p={m.f_pvalue:.4e}")
        for var in ['distance', 'p_mode_baseline', 'p_irr_baseline']:
            if var in m.params:
                print(f"    {var:25s}: β={m.params[var]:+.4f}, p={m.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Failed: {e}")

    # ══════════════════════════════════════════════════════
    # REGRESSION 2: DV = persisted (Logistic, committed only)
    # ══════════════════════════════════════════════════════
    committed_df = df_m[(df_m['committed_d1'] == 1)].copy()
    if 'persisted_d2' in committed_df.columns and len(committed_df) > 10:
        print(f"\n  {'─'*60}")
        print(f"  REG 2: DV = persisted_d2 (Logistic, committed only, N={len(committed_df)})")
        print(f"  {'─'*60}")
        try:
            m = smf.logit('persisted_d2 ~ distance + p_mode_baseline + d1_commitment_strength',
                           data=committed_df).fit(disp=0)
            print(f"  Pseudo R² = {m.prsquared:.4f}")
            for var in ['distance', 'p_mode_baseline', 'd1_commitment_strength']:
                if var in m.params:
                    or_val = np.exp(m.params[var])
                    print(f"    {var:25s}: β={m.params[var]:+.4f}, OR={or_val:.3f}, "
                          f"z={m.tvalues[var]:+.3f}, p={m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"    Failed: {e}")
    else:
        print(f"\n  REG 2: Skipped (N committed = {len(committed_df)})")

    # ══════════════════════════════════════════════════════
    # REGRESSION 3a: DV = residual ΔP(irr) after correction (OLS)
    # ══════════════════════════════════════════════════════
    df_corr = df_m.dropna(subset=['delta_p_irr_d2']).copy()
    if len(df_corr) > 10:
        print(f"\n  {'─'*60}")
        print(f"  REG 3a: DV = delta_p_irr_d2 (residual, OLS, N={len(df_corr)})")
        print(f"  {'─'*60}")
        try:
            m = smf.ols('delta_p_irr_d2 ~ distance + p_mode_baseline + committed_d1',
                          data=df_corr).fit()
            print(f"  R² = {m.rsquared:.4f}, F({m.df_model:.0f},{m.df_resid:.0f})={m.fvalue:.2f}, "
                  f"p={m.f_pvalue:.4e}")
            for var in ['distance', 'p_mode_baseline', 'committed_d1']:
                if var in m.params:
                    print(f"    {var:25s}: β={m.params[var]:+.4f}, p={m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"    Failed: {e}")

        # ══════════════════════════════════════════════════
        # REGRESSION 3b: DV = W₁(D1→D2) correction magnitude (OLS)
        # ══════════════════════════════════════════════════
        print(f"\n  {'─'*60}")
        print(f"  REG 3b: DV = w1_d1_d2 (correction magnitude, OLS, N={len(df_corr)})")
        print(f"  {'─'*60}")
        try:
            m = smf.ols('w1_d1_d2 ~ distance + p_mode_baseline + committed_d1',
                          data=df_corr).fit()
            print(f"  R² = {m.rsquared:.4f}, F({m.df_model:.0f},{m.df_resid:.0f})={m.fvalue:.2f}, "
                  f"p={m.f_pvalue:.4e}")
            for var in ['distance', 'p_mode_baseline', 'committed_d1']:
                if var in m.params:
                    print(f"    {var:25s}: β={m.params[var]:+.4f}, p={m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"    Failed: {e}")

        # ══════════════════════════════════════════════════
        # REGRESSION 3c: DV = W₁(baseline→D2) total residual (OLS)
        # ══════════════════════════════════════════════════
        print(f"\n  {'─'*60}")
        print(f"  REG 3c: DV = w1_bl_d2 (total residual displacement, OLS, N={len(df_corr)})")
        print(f"  {'─'*60}")
        try:
            m = smf.ols('w1_bl_d2 ~ distance + p_mode_baseline + committed_d1',
                          data=df_corr).fit()
            print(f"  R² = {m.rsquared:.4f}, F({m.df_model:.0f},{m.df_resid:.0f})={m.fvalue:.2f}, "
                  f"p={m.f_pvalue:.4e}")
            for var in ['distance', 'p_mode_baseline', 'committed_d1']:
                if var in m.params:
                    print(f"    {var:25s}: β={m.params[var]:+.4f}, p={m.pvalues[var]:.4e}")
        except Exception as e:
            print(f"    Failed: {e}")

        # ── T-test: committed vs uncommitted residual ──
        comm = df_corr[df_corr['committed_d1'] == 1]['delta_p_irr_d2']
        uncomm = df_corr[df_corr['committed_d1'] == 0]['delta_p_irr_d2']
        if len(comm) > 1 and len(uncomm) > 1:
            t, p = ttest_ind(comm, uncomm, equal_var=False)
            d_eff = cohens_d(comm.values, uncomm.values)
            print(f"\n  T-test: Committed vs Uncommitted residual ΔP(irr):")
            print(f"    Committed:   M={comm.mean():+.3f} (SD={comm.std():.3f}, n={len(comm)})")
            print(f"    Uncommitted: M={uncomm.mean():+.3f} (SD={uncomm.std():.3f}, n={len(uncomm)})")
            print(f"    t={t:.3f}, p={p:.4e}, Cohen's d={d_eff:.3f}")
    else:
        print(f"\n  REG 3: Skipped (no correction data)")

GPT-4o: 145 trials, committed=60 (41.4%)
DeepSeek-V3.2: FAILED — 'committed_d1'
Qwen-2.5: FAILED — 'committed_d1'
Phi-4: FAILED — 'committed_d1'

Total: 145 trials across 1 models

══════════════════════════════════════════════════════════════════════
  GPT-4o (N=145)
══════════════════════════════════════════════════════════════════════

  Descriptives:
    Commitment rate: 41.4%
    Mean ΔP(irr) at D1: +0.297
    Mean W₁(baseline→D1): 0.854
    Persistence (of committed): 53.3% (32/60)
    Mean residual ΔP(irr) at D2: +0.119

  ────────────────────────────────────────────────────────────
  REG 1a: DV = committed_d1 (Logistic)
  ────────────────────────────────────────────────────────────
  Pseudo R² = 0.0244, N = 145
    distance                 : β=-0.1891, OR=0.828, z=-0.728, p=4.6643e-01
    p_mode_baseline          : β=+1.6970, OR=5.458, z=+1.376, p=1.6869e-01
    p_irr_baseline           : β=+3.2617, OR=26.093, z=+1.513, p=1.3038e-01

  ──────────────────────────────────────────

In [6]:
"""
═══════════════════════════════════════════════════════════════
Study 3: Coalition Structure Effects on Opinion Stability
═══════════════════════════════════════════════════════════════

RESEARCH QUESTIONS:
  RQ1: Does coalition ratio (support vs opposition) affect opinion stability?
  RQ2: Does opposition extremity modulate the effect?
  RQ3: Does initial position interact with coalition structure?

DESIGN:
  - 4 support ratios: 3:0, 2:1, 1:2, 0:3
  - Opposition extremity: mild / moderate / extreme
  - 7-point scale (positions 1–7)

DVs:
  Primary:   switched (binary), ΔP(initial position)
  Secondary: W₁ (total distributional shift), shift magnitude

CONTROLS:
  - p_mode_baseline: prior commitment strength
  - base_extremity: |base_choice − 4|
  NOTE: mean_peer_distance NOT used — confounded with ratio (r = −0.78)

Split into cells at "# ── CELL ──" markers.
"""


# ── CELL: code — Setup & Data Loading ──

import json, re
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import f_oneway, ttest_ind, chi2_contingency, spearmanr
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

def get_dist7(probs_dict):
    d = np.array([float(probs_dict.get(str(i), 0) or 0) for i in range(1, 8)])
    return d / d.sum() if d.sum() > 0 else d

def wasserstein_1d(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    if p.sum() > 0: p /= p.sum()
    if q.sum() > 0: q /= q.sum()
    return np.sum(np.abs(np.cumsum(p)[:-1] - np.cumsum(q)[:-1]))

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    s = np.sqrt(((nx-1)*np.std(x,ddof=1)**2 + (ny-1)*np.std(y,ddof=1)**2) / (nx+ny-2))
    return (np.mean(x) - np.mean(y)) / s if s > 0 else 0

def parse_opp_extremity(condition):
    m = re.search(r'opp\d?_(\w+?)(?:_n4|$)', condition)
    return m.group(1) if m else None

# ══════════════════════════════════════════
# UPDATE THESE PATHS
# ══════════════════════════════════════════
STUDY3_MODELS = {
    "GPT-4o": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_gpt.jsonl",
    "DeepSeek-V3.2": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/outputs/study3_deepseek/summaries_deepseek.jsonl",
    "Phi-4": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_phi.jsonl",
    "Qwen-2.5": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_qwen.jsonl",
}

def build_study3_df(path, model_name):
    with open(path) as f:
        rows = [json.loads(line) for line in f if line.strip()]

    records = []
    for r in rows:
        if r.get("final_choice") is None or r.get("final_dist_probs") is None:
            continue

        # Skip rows with None inside probability dicts
        if None in r["baseline_dist_probs"].values() or None in r["final_dist_probs"].values():
            continue
        base_dist = get_dist7(r["baseline_dist_probs"])
        final_dist = get_dist7(r["final_dist_probs"])
        bc, fc = r["base_choice"], r["final_choice"]

        records.append({
            "model": model_name,
            "qid": r["qid"],
            "condition": r["condition"],
            "ratio": r["ratio"],
            "base_choice": bc,
            "final_choice": fc,
            "base_extremity": abs(bc - 4),
            "p_mode_baseline": base_dist[bc - 1],
            "opp_extremity": parse_opp_extremity(r["condition"]),
            "n_opposers": sum(1 for rl in r["peer_roles"] if rl == "corrector"),
            # DVs
            "switched": int(fc != bc),
            "shift_magnitude": abs(fc - bc),
            "delta_p_initial": final_dist[bc - 1] - base_dist[bc - 1],
            "w1": wasserstein_1d(base_dist, final_dist),
        })
    return pd.DataFrame(records)

s3_frames = []
for name, path in STUDY3_MODELS.items():
    try:
        df = build_study3_df(path, name)
        s3_frames.append(df)
        print(f"{name}: {len(df)} trials, switched={df['switched'].mean()*100:.1f}%")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s3 = pd.concat(s3_frames, ignore_index=True)
df_s3['n_support'] = df_s3['ratio'].map({'3:0': 3, '2:1': 2, '1:2': 1, '0:3': 0})

print(f"\nTotal: {len(df_s3)} trials, {df_s3['model'].nunique()} model(s)")
print(df_s3.groupby(['model', 'ratio'])['switched'].agg(['mean', 'count']).round(3))


# ── CELL: markdown ──
# ## 3.1 RQ1: Coalition Ratio → Opinion Stability


# ── CELL: code — RQ1 ──

print("="*70)
print("RQ1: DOES COALITION RATIO AFFECT OPINION STABILITY?")
print("="*70)

for model_name in df_s3['model'].unique():
    df_m = df_s3[df_s3['model'] == model_name]

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)})")
    print(f"{'━'*60}")

    # (a) Chi-square: switch rate ~ ratio
    ct = pd.crosstab(df_m['ratio'], df_m['switched'])
    chi2, p, dof, _ = chi2_contingency(ct)
    V = np.sqrt(chi2 / (len(df_m) * (min(ct.shape) - 1)))

    print(f"\n  (a) Chi-square: switch rate ~ ratio")
    print(f"      χ²({dof}) = {chi2:.2f}, p = {p:.2e}, Cramér's V = {V:.3f}")
    for r in ["3:0", "2:1", "1:2", "0:3"]:
        sub = df_m[df_m['ratio'] == r]
        print(f"      {r}: {sub['switched'].mean()*100:5.1f}% (n={len(sub)})")

    # (b) Logistic: switched ~ n_support + controls
    print(f"\n  (b) Logistic: switched ~ n_support + p_mode_baseline + base_extremity")
    m = smf.logit('switched ~ n_support + p_mode_baseline + base_extremity',
                  data=df_m).fit(disp=0)
    print(f"      Pseudo R² = {m.prsquared:.4f}, N = {int(m.nobs)}")
    for var in ['n_support', 'p_mode_baseline', 'base_extremity']:
        OR = np.exp(m.params[var])
        print(f"      {var:20s}: β={m.params[var]:+.3f}, OR={OR:.3f}, "
              f"z={m.tvalues[var]:+.2f}, p={m.pvalues[var]:.2e}")

    # (c) OLS: ΔP(initial) ~ n_support + controls
    print(f"\n  (c) OLS: delta_p_initial ~ n_support + p_mode_baseline + base_extremity")
    m = smf.ols('delta_p_initial ~ n_support + p_mode_baseline + base_extremity',
                data=df_m).fit()
    print(f"      R² = {m.rsquared:.4f}, F({m.df_model:.0f},{m.df_resid:.0f}) = {m.fvalue:.2f}, p = {m.f_pvalue:.2e}")
    for var in ['n_support', 'p_mode_baseline', 'base_extremity']:
        print(f"      {var:20s}: β={m.params[var]:+.4f}, t={m.tvalues[var]:+.2f}, p={m.pvalues[var]:.2e}")

    # (d) ANOVA + Tukey on W₁
    print(f"\n  (d) ANOVA: W₁ ~ ratio + Tukey HSD")
    groups = {r: df_m[df_m['ratio']==r]['w1'].values for r in ["3:0","2:1","1:2","0:3"]}
    F, p = f_oneway(*groups.values())
    k, n_tot = len(groups), sum(len(g) for g in groups.values())
    eta_sq = (F*(k-1)) / (F*(k-1) + (n_tot-k))
    print(f"      F({k-1},{n_tot-k}) = {F:.2f}, p = {p:.2e}, η² = {eta_sq:.4f}")
    for r in ["3:0","2:1","1:2","0:3"]:
        g = groups[r]
        print(f"      {r}: M={g.mean():.4f}, SD={g.std():.4f}")
    print(pairwise_tukeyhsd(df_m['w1'], df_m['ratio'], alpha=0.05).summary())


# ── CELL: markdown ──
# ## 3.2 RQ2: Opposition Extremity Modulation


# ── CELL: code — RQ2 ──

print("="*70)
print("RQ2: DOES OPPOSITION EXTREMITY MODULATE THE EFFECT?")
print("="*70)

for model_name in df_s3['model'].unique():
    df_m = df_s3[(df_s3['model'] == model_name) &
                  (df_s3['ratio'].isin(['0:3', '1:2']))].copy()
    df_m = df_m.dropna(subset=['opp_extremity'])
    df_m['opp_ext_code'] = df_m['opp_extremity'].map({'mild': 0, 'moderate': 1, 'extreme': 2})

    print(f"\n{'━'*60}")
    print(f"  {model_name} — Opposition trials (N={len(df_m)})")
    print(f"{'━'*60}")

    # (a) Descriptives
    print(f"\n  (a) Descriptives by opposition extremity:")
    for ext in ['mild', 'moderate', 'extreme']:
        sub = df_m[df_m['opp_extremity'] == ext]
        print(f"      {ext:10s}: switch={sub['switched'].mean()*100:.1f}%, "
              f"ΔP(init)={sub['delta_p_initial'].mean():+.3f}, "
              f"W₁={sub['w1'].mean():.3f} (n={len(sub)})")

    # (b) Chi-square: switch ~ extremity
    ct = pd.crosstab(df_m['opp_extremity'], df_m['switched'])
    chi2, p, dof, _ = chi2_contingency(ct)
    V = np.sqrt(chi2 / (len(df_m) * (min(ct.shape)-1)))
    print(f"\n  (b) Chi-square: switch ~ opp_extremity")
    print(f"      χ²({dof}) = {chi2:.2f}, p = {p:.2e}, V = {V:.3f}")

    # (c) Logistic: tests extremity ABOVE AND BEYOND the ratio (n_opposers)
    print(f"\n  (c) Logistic: switched ~ opp_ext_code + n_opposers + p_mode_baseline + base_extremity")
    try:
        m = smf.logit('switched ~ opp_ext_code + n_opposers + p_mode_baseline + base_extremity',
                      data=df_m).fit(disp=0)
        print(f"      Pseudo R² = {m.prsquared:.4f}")
        for var in ['opp_ext_code', 'n_opposers', 'p_mode_baseline', 'base_extremity']:
            OR = np.exp(m.params[var])
            print(f"      {var:20s}: β={m.params[var]:+.3f}, OR={OR:.3f}, "
                  f"z={m.tvalues[var]:+.2f}, p={m.pvalues[var]:.2e}")
    except Exception as e:
        print(f"      Failed: {e}")

    # (d) OLS: ΔP(initial) ~ extremity + ratio + controls
    print(f"\n  (d) OLS: delta_p_initial ~ opp_ext_code + n_opposers + p_mode_baseline + base_extremity")
    try:
        m = smf.ols('delta_p_initial ~ opp_ext_code + n_opposers + p_mode_baseline + base_extremity',
                    data=df_m).fit()
        print(f"      R² = {m.rsquared:.4f}, F = {m.fvalue:.2f}, p = {m.f_pvalue:.2e}")
        for var in ['opp_ext_code', 'n_opposers', 'p_mode_baseline', 'base_extremity']:
            print(f"      {var:20s}: β={m.params[var]:+.4f}, t={m.tvalues[var]:+.2f}, p={m.pvalues[var]:.2e}")
    except Exception as e:
        print(f"      Failed: {e}")


# ── CELL: markdown ──
# ## 3.3 RQ3: Initial Position × Coalition Interaction


# ── CELL: code — RQ3 ──

print("="*70)
print("RQ3: DOES INITIAL POSITION MODERATE THE COALITION EFFECT?")
print("="*70)

for model_name in df_s3['model'].unique():
    df_m = df_s3[df_s3['model'] == model_name].copy()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)})")
    print(f"{'━'*60}")

    # (a) Logistic with interaction
    print(f"\n  (a) Logistic: switched ~ n_support * base_extremity + p_mode_baseline")
    m = smf.logit('switched ~ n_support * base_extremity + p_mode_baseline',
                  data=df_m).fit(disp=0)
    print(f"      Pseudo R² = {m.prsquared:.4f}")
    for var in m.params.index:
        if var == 'Intercept': continue
        OR = np.exp(m.params[var])
        print(f"      {var:35s}: β={m.params[var]:+.3f}, OR={OR:.3f}, "
              f"z={m.tvalues[var]:+.2f}, p={m.pvalues[var]:.2e}")

    # (b) OLS with interaction
    print(f"\n  (b) OLS: delta_p_initial ~ n_support * base_extremity + p_mode_baseline")
    m = smf.ols('delta_p_initial ~ n_support * base_extremity + p_mode_baseline',
                data=df_m).fit()
    print(f"      R² = {m.rsquared:.4f}, Adj R² = {m.rsquared_adj:.4f}")
    for var in m.params.index:
        if var == 'Intercept': continue
        print(f"      {var:35s}: β={m.params[var]:+.4f}, t={m.tvalues[var]:+.2f}, p={m.pvalues[var]:.2e}")

    # (c) Stability table
    print(f"\n  (c) Stability rate (%) by position × ratio:")
    header = f"  {'Pos':>4s}"
    for r in ["3:0", "2:1", "1:2", "0:3"]:
        header += f"  {r:>8s}"
    print(header)
    for pos in range(1, 8):
        row = f"  {pos:>4d}"
        for r in ["3:0", "2:1", "1:2", "0:3"]:
            sub = df_m[(df_m['base_choice']==pos) & (df_m['ratio']==r)]
            if len(sub) >= 5:
                row += f"  {(1-sub['switched'].mean())*100:>7.1f}%"
            else:
                row += f"  {'—':>8s}"
        print(row)


# ── CELL: markdown ──
# ## 3.4 Summary: Monotonic Trends


# ── CELL: code — Trends ──

print("="*70)
print("MONOTONIC TRENDS: Spearman ρ(n_support, DV)")
print("="*70)

for model_name in df_s3['model'].unique():
    df_m = df_s3[df_s3['model'] == model_name]
    print(f"\n  {model_name}:")
    for dv, label in [
        ('switched',        'Switch rate'),
        ('delta_p_initial', 'ΔP(initial)'),
        ('w1',              'W₁'),
        ('shift_magnitude', 'Shift magnitude'),
    ]:
        rho, p = spearmanr(df_m['n_support'], df_m[dv])
        sig = '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else 'ns'))
        print(f"    {label:20s}: ρ = {rho:+.3f}, p = {p:.2e} {sig}")

GPT-4o: 5692 trials, switched=22.6%
DeepSeek-V3.2: 5508 trials, switched=31.2%
Phi-4: 5694 trials, switched=53.0%
Qwen-2.5: 5694 trials, switched=61.8%

Total: 22588 trials, 4 model(s)
                      mean  count
model         ratio              
DeepSeek-V3.2 0:3    0.523    682
              1:2    0.317   2702
              2:1    0.238   1824
              3:0    0.247    300
GPT-4o        0:3    0.807    701
              1:2    0.119   2807
              2:1    0.173   1872
              3:0    0.212    312
Phi-4         0:3    0.893    702
              1:2    0.600   2808
              2:1    0.338   1872
              3:0    0.244    312
Qwen-2.5      0:3    0.749    702
              1:2    0.668   2808
              2:1    0.544   1872
              3:0    0.321    312
RQ1: DOES COALITION RATIO AFFECT OPINION STABILITY?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  GPT-4o (N=5692)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  (a) Chi

In [ ]:
STUDY3_MODELS = {
    "GPT-4o": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_gpt.jsonl",
    "DeepSeek-V3.2": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/outputs/study3_deepseek/summaries_deepseek.jsonl",
    "Phi-4": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_phi.jsonl",
    "Qwen-2.5": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_qwen.jsonl",
}

def build_study3_df(path, model_name):
    with open(path) as f:
        rows = [json.loads(line) for line in f if line.strip()]

    records = []
    for r in rows:
        if r.get("final_choice") is None or r.get("final_dist_probs") is None:
            continue

        # Skip rows with None inside probability dicts
        if None in r["baseline_dist_probs"].values() or None in r["final_dist_probs"].values():
            continue
        base_dist = get_dist7(r["baseline_dist_probs"])
        final_dist = get_dist7(r["final_dist_probs"])
        bc, fc = r["base_choice"], r["final_choice"]

        records.append({
            "model": model_name,
            "qid": r["qid"],
            "condition": r["condition"],
            "ratio": r["ratio"],
            "base_choice": bc,
            "final_choice": fc,
            "base_extremity": abs(bc - 4),
            "p_mode_baseline": base_dist[bc - 1],
            "opp_extremity": parse_opp_extremity(r["condition"]),
            "n_opposers": sum(1 for rl in r["peer_roles"] if rl == "corrector"),
            # DVs
            "switched": int(fc != bc),
            "shift_magnitude": abs(fc - bc),
            "delta_p_initial": final_dist[bc - 1] - base_dist[bc - 1],
            "w1": wasserstein_1d(base_dist, final_dist),
        })
    return pd.DataFrame(records)

s3_frames = []
for name, path in STUDY3_MODELS.items():
    try:
        df = build_study3_df(path, name)
        s3_frames.append(df)
        print(f"{name}: {len(df)} trials, switched={df['switched'].mean()*100:.1f}%")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s3 = pd.concat(s3_frames, ignore_index=True)
df_s3['n_support'] = df_s3['ratio'].map({'3:0': 3, '2:1': 2, '1:2': 1, '0:3': 0})

print(f"\nTotal: {len(df_s3)} trials, {df_s3['model'].nunique()} model(s)")
print(df_s3.groupby(['model', 'ratio'])['switched'].agg(['mean', 'count']).round(3))

In [ ]:
print("Columns in df_s3:")
print(list(df_s3.columns))
print(f"\nShape: {df_s3.shape}")
print(f"\nSample row:")
print(df_s3.iloc[0].to_dict())

In [ ]:
print("Unique conditions (sample):")
print(df_s3['condition'].unique()[:30])
print(f"\nTotal unique conditions: {df_s3['condition'].nunique()}")

In [ ]:
def parse_condition_distance(row):
    """Extract average peer distance from condition string."""
    cond = str(row['condition'])
    base = row['base_choice']
    
    # Distance labels → approximate Likert distances
    dist_map = {'match': 0, 'close': 1, 'moderate': 2, 'extreme': 3, 'mild': 1.5}
    
    # Parse: sup3_match, opp3_mild_n4L, sup2_match_opp1_mild_n4L, etc.
    # Remove the n4L/n4R/n4Split suffix
    clean = cond
    for suffix in ['_n4L', '_n4R', '_n4Split']:
        clean = clean.replace(suffix, '')
    
    parts = clean.split('_')
    distances = []
    
    i = 0
    while i < len(parts):
        p = parts[i]
        if p.startswith('sup'):
            n = int(p[3:]) if len(p) > 3 and p[3:].isdigit() else 1
            if i + 1 < len(parts) and parts[i+1] in dist_map:
                d = dist_map[parts[i+1]]
                distances.extend([d] * n)
                i += 2
            else:
                i += 1
        elif p.startswith('opp'):
            n = int(p[3:]) if len(p) > 3 and p[3:].isdigit() else 1
            if i + 1 < len(parts) and parts[i+1] in dist_map:
                d = dist_map[parts[i+1]]
                distances.extend([d] * n)
                i += 2
            else:
                i += 1
        else:
            i += 1
    
    if distances:
        return pd.Series({
            'avg_dist_peers': np.mean(distances),
            'max_dist_peers': np.max(distances),
            'peer_spread': np.std(distances) if len(distances) > 1 else 0,
            'n_peers': len(distances),
        })
    else:
        return pd.Series({
            'avg_dist_peers': np.nan,
            'max_dist_peers': np.nan,
            'peer_spread': np.nan,
            'n_peers': 0,
        })

# Compute
print("Parsing conditions...")
dist_df = df_s3.apply(parse_condition_distance, axis=1)
df_s3 = pd.concat([df_s3, dist_df], axis=1)

# Verify
print(f"\navg_dist_peers stats:")
print(df_s3['avg_dist_peers'].describe())
print(f"\nVerification — parsed conditions:")
for cond in ['sup3_match', 'sup3_extreme', 'opp3_mild_n4L', 'opp3_extreme_n4R',
             'sup2_match_opp1_mild_n4L', 'sup2_moderate_opp1_extreme_n4R',
             'sup1_close_opp2_moderate_n4L']:
    subset = df_s3[df_s3['condition'] == cond]
    if len(subset) > 0:
        r = subset.iloc[0]
        print(f"  {cond:45s} → avg_dist={r['avg_dist_peers']:.2f}, "
              f"spread={r['peer_spread']:.2f}, n_peers={int(r['n_peers'])}")

In [ ]:
from scipy.stats import f as f_dist
df_s3_clean = df_s3.dropna(subset=['avg_dist_peers', 'delta_p_initial', 
                                    'n_support', 'p_mode_baseline', 'base_extremity'])
print(f"Clean N: {len(df_s3_clean)}")

for model_name in df_s3_clean['model'].unique():
    df_m = df_s3_clean[df_s3_clean['model'] == model_name].copy()
    
    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)})")
    print(f"{'━'*60}")
    
    # ── Model 1: Main effects only ──
    print("\n  --- Model 1: Main effects ---")
    m1 = smf.ols('delta_p_initial ~ n_support + avg_dist_peers + p_mode_baseline + base_extremity',
                  data=df_m).fit()
    print(f"  R² = {m1.rsquared:.4f}")
    for var in ['n_support', 'avg_dist_peers', 'p_mode_baseline', 'base_extremity']:
        print(f"    {var:25s}: β={m1.params[var]:+.4f}, t={m1.tvalues[var]:+.2f}, p={m1.pvalues[var]:.4e}")
    
    # ── Model 2: + Coalition × Avg Distance interaction ──
    print("\n  --- Model 2: + n_support × avg_dist_peers ---")
    m2 = smf.ols('delta_p_initial ~ n_support * avg_dist_peers + p_mode_baseline + base_extremity',
                  data=df_m).fit()
    print(f"  R² = {m2.rsquared:.4f}")
    for var in m2.params.index:
        if var == 'Intercept': continue
        print(f"    {var:40s}: β={m2.params[var]:+.4f}, t={m2.tvalues[var]:+.2f}, p={m2.pvalues[var]:.4e}")
    
    # F-test: does interaction improve fit?
    ssr1, ssr2 = m1.ssr, m2.ssr
    df_diff = m2.df_model - m1.df_model
    f_stat = ((ssr1 - ssr2) / df_diff) / (ssr2 / m2.df_resid)
    p_val = 1 - f_dist.cdf(f_stat, df_diff, m2.df_resid)
    print(f"\n  Interaction improvement: F({df_diff},{m2.df_resid:.0f})={f_stat:.3f}, p={p_val:.4e}, ΔR²={m2.rsquared - m1.rsquared:.4f}")
    
    # ── Model 3: + Peer spread interaction ──
    if df_m['peer_spread'].std() > 0:
        print("\n  --- Model 3: + n_support × peer_spread ---")
        m3 = smf.ols('delta_p_initial ~ n_support * avg_dist_peers + n_support * peer_spread + p_mode_baseline + base_extremity',
                      data=df_m).fit()
        print(f"  R² = {m3.rsquared:.4f}")
        for var in m3.params.index:
            if var == 'Intercept': continue
            print(f"    {var:40s}: β={m3.params[var]:+.4f}, t={m3.tvalues[var]:+.2f}, p={m3.pvalues[var]:.4e}")
    
    # ── Model 4: Logistic — switching ──
    print("\n  --- Model 4: Logistic — switched ~ n_support × avg_dist ---")
    try:
        m4 = smf.logit('switched ~ n_support * avg_dist_peers + p_mode_baseline + base_extremity',
                        data=df_m).fit(disp=0)
        print(f"  Pseudo R² = {m4.prsquared:.4f}")
        for var in m4.params.index:
            if var == 'Intercept': continue
            or_val = np.exp(m4.params[var])
            print(f"    {var:40s}: β={m4.params[var]:+.4f}, OR={or_val:.3f}, p={m4.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Failed: {e}")
    
    # ── Simple slopes ──
    if 'n_support:avg_dist_peers' in m2.params:
        print("\n  --- Simple slopes: n_support effect at different distances ---")
        b_sup = m2.params['n_support']
        b_int = m2.params['n_support:avg_dist_peers']
        mean_d = df_m['avg_dist_peers'].mean()
        sd_d = df_m['avg_dist_peers'].std()
        for label, val in [('close (−1SD)', mean_d - sd_d), ('mean', mean_d), ('far (+1SD)', mean_d + sd_d)]:
            slope = b_sup + b_int * val
            print(f"    At dist={val:.2f} ({label}): support effect = {slope:+.4f}")
    
    # ── Descriptive crosstab ──
    print("\n  --- ΔP(initial) by ratio × distance tercile ---")
    df_m['dist_tercile'] = pd.qcut(df_m['avg_dist_peers'], 3, labels=['close', 'medium', 'far'], duplicates='drop')
    ct = df_m.groupby(['ratio', 'dist_tercile'])['delta_p_initial'].mean().unstack('dist_tercile')
    print(ct.to_string(float_format=lambda x: f"{x:+.3f}"))

In [ ]:

MODELS = {
    "GPT-4o": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/baseline_outputs_explicit_format.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/study1_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/study2_v3_experimental_outputs.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/gpt4o_study2_additional_results.jsonl",
    },
    "DeepSeek-V3.2": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/outputs/deepseek_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/outputs/deepseek_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/deepseek_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/deepseek_study2_additional_results.jsonl",
    },

    "Olmo-3.5": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/olmo_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/olmo_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/olmo_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/olmo_study2_additional_results.jsonl",
    },

    "qwen": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/qwen_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/qwen_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/qwen_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/qwen_study2_additional_results.jsonl",

    },

    'Phi-4': {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/phi4_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/phi4_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/phi4_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/phi4_study2_additional_results.jsonl",

}
}

# Study 3 path (if separate)
STUDY3_MODELS = {
    "GPT-4o": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_gpt.jsonl",
    "DeepSeek-V3.2": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/outputs/study3_deepseek/summaries_deepseek.jsonl",
    "Phi-4": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_phi.jsonl",
    "Qwen-2.5": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_qwen.jsonl",
} # or build from raw

print("Paths defined. Update before running.")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 3: Build Study 1 DataFrame
# ══════════════════════════════════════════════════════════

def load_baseline(path):
    bl = {}
    for obj in load_jsonl(path):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        md = obj.get("metadata", {})
        qid = md.get("qid") or obj.get("custom_id", "").split("|")[0]
        probs = resp.get("probs", {})
        if qid and probs:
            dist = get_dist7(probs)
            mode = int(np.argmax(dist)) + 1
            bl[qid] = {"dist": dist, "mode": mode, "p_mode": dist[mode-1]}
    return bl

def build_study1_df(model_name, paths):
    bl = load_baseline(paths["baseline"])
    rows = []
    for obj in load_jsonl(paths["study1_full"]):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        probs = resp.get("probs", {})
        if not probs: continue
        md = obj.get("metadata", {})
        qid = md.get("qid") or obj.get("custom_id", "").split("|")[0]
        if qid not in bl: continue

        post_dist = get_dist7(probs)
        base = bl[qid]
        mode = base["mode"]

        cue_target = md.get("cue_target")
        if cue_target is None or cue_target == "none": continue
        cue_target = int(cue_target)

        cue_type = str(md.get("cue_direction", md.get("condition", ""))).lower()
        if "oppose" in cue_type: cue_type = "oppose"
        elif "pull" in cue_type: cue_type = "pull"
        elif "reinforce" in cue_type: cue_type = "reinforce"
        elif "memory" in cue_type: cue_type = "memory_only"

        distance = md.get("cue_distance")
        if distance is not None and distance != "none":
            distance = int(distance)
        else:
            distance = abs(cue_target - mode)

        rows.append({
            "model": model_name, "qid": qid, "mode": mode,
            "p_mode_baseline": base["p_mode"],
            "extremity": abs(mode - 4),
            "cue_target": cue_target, "cue_type": cue_type,
            "distance": distance,
            "dp_prior": post_dist[mode-1] - base["dist"][mode-1],
            "dp_cue": post_dist[cue_target-1] - base["dist"][cue_target-1],
            "w1": wasserstein_1d(base["dist"], post_dist),
        })
    return pd.DataFrame(rows)

# Build
s1_frames = []
for name, paths in MODELS.items():
    try:
        df = build_study1_df(name, paths)
        s1_frames.append(df)
        print(f"{name}: {len(df)} trials, {df[df['cue_type']=='oppose'].shape[0]} oppose")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s1 = pd.concat(s1_frames, ignore_index=True)
print(f"\nTotal Study 1: {len(df_s1)}")
print(df_s1.groupby(['model', 'cue_type']).size().unstack(fill_value=0))

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 4: Build Study 2 DataFrame
# ══════════════════════════════════════════════════════════

def build_study2_df(model_name, paths):
    bl = load_baseline(paths["baseline"])
    rows = []
    for obj in load_jsonl(paths["study2_additional"]):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        probs = resp.get("probs", {})
        if not probs: continue
        md = obj.get("metadata", {})
        qid = md.get("qid")
        if qid not in bl: continue

        post_dist = get_dist7(probs)
        irr = md["irrational_choice"]
        mode = md["mode_choice"]

        rows.append({
            "model": model_name, "qid": qid,
            "experiment": md.get("experiment"),
            "framing": md.get("framing"),
            "attribution": md.get("attribution"),
            "stage": md.get("stage"),
            "irr": irr, "mode": mode,
            "distance": md.get("distance"),
            "p_mode_baseline": bl[qid]["p_mode"],
            "p_irr": post_dist[irr-1],
            "p_mode": post_dist[mode-1],
            "committed": int(post_dist[irr-1] > post_dist[mode-1]),
            "delta_p_irr": post_dist[irr-1] - bl[qid]["dist"][irr-1],
            "w1": wasserstein_1d(bl[qid]["dist"], post_dist),
        })
    return pd.DataFrame(rows)

s2_frames = []
for name, paths in MODELS.items():
    try:
        df = build_study2_df(name, paths)
        s2_frames.append(df)
        print(f"{name}: {len(df)} trials")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s2 = pd.concat(s2_frames, ignore_index=True)
print(f"\nTotal Study 2: {len(df_s2)}")

In [ ]:
def build_study3_df(path, model_name):
    with open(path) as f:
        rows = [json.loads(line) for line in f if line.strip()]

    records = []
    for r in rows:
        if r.get("final_choice") is None or r.get("final_dist_probs") is None:
            continue

        # Skip rows with None inside probability dicts
        if None in r["baseline_dist_probs"].values() or None in r["final_dist_probs"].values():
            continue
        base_dist = get_dist7(r["baseline_dist_probs"])
        final_dist = get_dist7(r["final_dist_probs"])
        bc, fc = r["base_choice"], r["final_choice"]

        records.append({
            "model": model_name,
            "qid": r["qid"],
            "condition": r["condition"],
            "ratio": r["ratio"],
            "base_choice": bc,
            "final_choice": fc,
            "base_extremity": abs(bc - 4),
            "p_mode_baseline": base_dist[bc - 1],
            "opp_extremity": parse_opp_extremity(r["condition"]),
            "n_opposers": sum(1 for rl in r["peer_roles"] if rl == "corrector"),
            # DVs
            "switched": int(fc != bc),
            "shift_magnitude": abs(fc - bc),
            "delta_p_initial": final_dist[bc - 1] - base_dist[bc - 1],
            "w1": wasserstein_1d(base_dist, final_dist),
        })
    return pd.DataFrame(records)

s3_frames = []
for name, path in STUDY3_MODELS.items():
    try:
        df = build_study3_df(path, name)
        s3_frames.append(df)
        print(f"{name}: {len(df)} trials, switched={df['switched'].mean()*100:.1f}%")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s3 = pd.concat(s3_frames, ignore_index=True)
df_s3['n_support'] = df_s3['ratio'].map({'3:0': 3, '2:1': 2, '1:2': 1, '0:3': 0})

print(f"\nTotal: {len(df_s3)} trials, {df_s3['model'].nunique()} model(s)")
print(df_s3.groupby(['model', 'ratio'])['switched'].agg(['mean', 'count']).round(3))

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 5: Study 1 — Corrected regressions (mixed-effects or cluster-robust)
# ══════════════════════════════════════════════════════════

print("="*70)
print("STUDY 1 — Primary: ΔP(cue target) ~ distance + distance² + controls")
print("Oppose trials only. Cluster-robust SEs by dilemma.")
print("="*70)

for model_name in df_s1['model'].unique():
    opp = df_s1[(df_s1['model'] == model_name) & (df_s1['cue_type'] == 'oppose')].copy()
    opp = opp.dropna(subset=['dp_cue', 'distance', 'p_mode_baseline'])
    opp['distance_sq'] = opp['distance'] ** 2
    n_dilemmas = opp['qid'].nunique()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(opp)}, {n_dilemmas} dilemmas)")
    print(f"{'━'*60}")

    use_mixed = HAS_MIXED
    if use_mixed:
        try:
            m_lin = smf.mixedlm('dp_cue ~ distance + p_mode_baseline + extremity',
                                 data=opp, groups=opp['qid']).fit(reml=True)
            m_quad = smf.mixedlm('dp_cue ~ distance + distance_sq + p_mode_baseline + extremity',
                                  data=opp, groups=opp['qid']).fit(reml=True)

            print("\n  --- Linear (mixed-effects, random intercept for dilemma) ---")
            print(f"  Log-lik: {m_lin.llf:.2f}")
            for var in ['distance', 'p_mode_baseline', 'extremity']:
                ci = m_lin.conf_int().loc[var]
                print(f"    {var:20s}: β={m_lin.fe_params[var]:+.4f}, "
                      f"SE={m_lin.bse_fe[var]:.4f}, z={m_lin.tvalues[var]:+.2f}, "
                      f"p={m_lin.pvalues[var]:.4e}, 95%CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")

            print(f"\n  --- Quadratic (mixed-effects) ---")
            print(f"  Log-lik: {m_quad.llf:.2f}")
            for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
                ci = m_quad.conf_int().loc[var]
                print(f"    {var:20s}: β={m_quad.fe_params[var]:+.4f}, "
                      f"SE={m_quad.bse_fe[var]:.4f}, z={m_quad.tvalues[var]:+.2f}, "
                      f"p={m_quad.pvalues[var]:.4e}, 95%CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")

            lr_stat = 2 * (m_quad.llf - m_lin.llf)
            lr_p = 1 - stats.chi2.cdf(lr_stat, df=1)
            print(f"\n  LR test (linear vs quadratic): χ²(1)={lr_stat:.3f}, p={lr_p:.4e}")

        except Exception as e:
            print(f"  Mixed-effects failed: {e}")
            print(f"  Falling back to cluster-robust SEs.")
            use_mixed = False

    if not use_mixed:
        m_lin = smf.ols('dp_cue ~ distance + p_mode_baseline + extremity',
                          data=opp).fit(cov_type='cluster', cov_kwds={'groups': opp['qid']})
        m_quad = smf.ols('dp_cue ~ distance + distance_sq + p_mode_baseline + extremity',
                           data=opp).fit(cov_type='cluster', cov_kwds={'groups': opp['qid']})

        print(f"\n  --- Linear (OLS, cluster-robust SEs) ---")
        print(f"  R² = {m_lin.rsquared:.4f}")
        for var in ['distance', 'p_mode_baseline', 'extremity']:
            ci = m_lin.conf_int().loc[var]
            print(f"    {var:20s}: β={m_lin.params[var]:+.4f}, SE={m_lin.bse[var]:.4f}, "
                  f"t={m_lin.tvalues[var]:+.2f}, p={m_lin.pvalues[var]:.4e}, "
                  f"95%CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")

        print(f"\n  --- Quadratic (OLS, cluster-robust SEs) ---")
        print(f"  R² = {m_quad.rsquared:.4f}")
        for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
            ci = m_quad.conf_int().loc[var]
            print(f"    {var:20s}: β={m_quad.params[var]:+.4f}, SE={m_quad.bse[var]:.4f}, "
                  f"t={m_quad.tvalues[var]:+.2f}, p={m_quad.pvalues[var]:.4e}, "
                  f"95%CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")

        from scipy.stats import f as f_dist
        ssr1, ssr2 = m_lin.ssr, m_quad.ssr
        f_stat = ((ssr1 - ssr2) / 1) / (ssr2 / m_quad.df_resid)
        f_p = 1 - f_dist.cdf(f_stat, 1, m_quad.df_resid)
        print(f"\n  Quadratic improvement: F(1,{m_quad.df_resid:.0f})={f_stat:.3f}, "
              f"p={f_p:.4e}, ΔR²={m_quad.rsquared - m_lin.rsquared:.4f}")

In [ ]:
print("="*70)
print("STUDY 2 ORIGINAL — Memory injection → Correction")
print("="*70)

for model_name, paths in MODELS.items():
    try:
        path = paths["study2_testing"]
        df_m = build_study2_orig(path, model_name)

def build_study2_orig(path, model_name):
    rows_raw = load_jsonl(path)
    bl = load_baseline(MODELS[model_name]["baseline"])

    records = {}
    for r in rows_raw:
        resp = r.get("response", {})
        if resp.get("status_code") != 200: continue
        md = r.get("metadata", {})
        qid = md.get("qid")
        irr = md.get("irrational_choice")
        condition = md.get("condition", "")
        if not qid or irr is None or qid not in bl: continue
        probs = resp.get("probs", {})
        if not probs: continue

        dist = get_dist7(probs)
        key = (qid, irr)
        if key not in records:
            records[key] = {
                "qid": qid, "irr": irr,
                "mode": bl[qid]["mode"],
                "distance": md.get("distance"),
                "p_mode_baseline": bl[qid]["p_mode"],
                "p_irr_baseline": bl[qid]["dist"][irr - 1],
            }
        records[key][f"{condition}_dist"] = dist

    rows_out = []
    for key, rec in records.items():
        if "memory_justify_dist" not in rec: continue
        d1 = rec["memory_justify_dist"]
        irr, mode = rec["irr"], rec["mode"]
        p_irr_bl = rec["p_irr_baseline"]
        p_mode_bl = rec["p_mode_baseline"]

        d1_p_irr = d1[irr - 1]
        d1_p_mode = d1[mode - 1]
        committed = d1_p_irr > d1_p_mode

        row = {
            "model": model_name, "qid": rec["qid"],
            "irr": irr, "mode": mode,
            "distance": rec.get("distance"),
            "p_mode_baseline": p_mode_bl,
            "p_irr_baseline": p_irr_bl,
            "committed_d1": int(committed),
            "delta_p_irr_d1": d1_p_irr - p_irr_bl,
            "d1_p_irr": d1_p_irr,
            "d1_commitment_strength": d1_p_irr - d1_p_mode,
        }

        if "memory_justify_correction_dist" in rec:
            d2 = rec["memory_justify_correction_dist"]
            d2_p_irr = d2[irr - 1]
            d2_p_mode = d2[mode - 1]
            row["persisted_d2"] = int(d2_p_irr > d2_p_mode)
            row["delta_p_irr_d2"] = d2_p_irr - p_irr_bl
            row["w1_d1_d2"] = wasserstein_1d(d1, d2)

        rows_out.append(row)
    return pd.DataFrame(rows_out)

# Run for all models
for model_name, paths in MODELS.items():
    try:
        df_m = build_study2_orig(paths["study2_testing"], model_name)
        df_m['distance'] = pd.to_numeric(df_m['distance'], errors='coerce')
        df_m['p_mode_baseline'] = pd.to_numeric(df_m['p_mode_baseline'], errors='coerce')
        df_m['p_irr_baseline'] = pd.to_numeric(df_m['p_irr_baseline'], errors='coerce')
        df_m = df_m.dropna(subset=['distance', 'p_mode_baseline'])
        n_dilemmas = df_m['qid'].nunique()

        print(f"\n{'━'*60}")
        print(f"  {model_name} (N={len(df_m)}, {n_dilemmas} dilemmas)")
        print(f"{'━'*60}")

        # Descriptives
        print(f"\n  Descriptives:")
        print(f"    Commitment rate: {df_m['committed_d1'].mean()*100:.1f}%")
        ci_lo, ci_hi = proportion_confint(int(df_m['committed_d1'].sum()), len(df_m), method='wilson')
        print(f"    95% CI: [{ci_lo*100:.1f}, {ci_hi*100:.1f}]")
        print(f"    Mean ΔP(irr) at D1: {df_m['delta_p_irr_d1'].mean():+.3f}")
        if 'persisted_d2' in df_m.columns:
            comm = df_m[df_m['committed_d1'] == 1]
            if len(comm) > 0:
                pers_rate = comm['persisted_d2'].mean() * 100
                ci_lo2, ci_hi2 = proportion_confint(int(comm['persisted_d2'].sum()), len(comm), method='wilson')
                print(f"    Persistence: {pers_rate:.1f}% [{ci_lo2*100:.1f}, {ci_hi2*100:.1f}] "
                      f"({int(comm['persisted_d2'].sum())}/{len(comm)})")
        if 'delta_p_irr_d2' in df_m.columns:
            print(f"    Mean residual ΔP(irr) at D2: {df_m['delta_p_irr_d2'].mean():+.3f}")

        # Reg 1: DV = ΔP(irr) at D1
        print(f"\n  --- Reg 1: DV = ΔP(irr) at D1, cluster-robust ---")
        try:
            m1 = smf.ols('delta_p_irr_d1 ~ distance + p_mode_baseline + p_irr_baseline',
                           data=df_m).fit(cov_type='cluster', cov_kwds={'groups': df_m['qid']})
            print(f"  R² = {m1.rsquared:.4f}")
            for var in ['distance', 'p_mode_baseline', 'p_irr_baseline']:
                ci = m1.conf_int().loc[var]
                print(f"    {var:25s}: β={m1.params[var]:+.4f}, t={m1.tvalues[var]:+.2f}, "
                      f"p={m1.pvalues[var]:.4e}, CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")
        except Exception as e:
            print(f"    Failed: {e}")

        # Reg 2: DV = committed (logistic)
        print(f"\n  --- Reg 2: DV = committed_d1 (logistic, cluster-robust) ---")
        try:
            m2 = smf.logit('committed_d1 ~ distance + p_mode_baseline + p_irr_baseline',
                             data=df_m).fit(disp=0,
                             cov_type='cluster', cov_kwds={'groups': df_m['qid']})
            print(f"  Pseudo R² = {m2.prsquared:.4f}")
            for var in ['distance', 'p_mode_baseline', 'p_irr_baseline']:
                or_val = np.exp(m2.params[var])
                ci = m2.conf_int().loc[var]
                or_ci = [np.exp(ci[0]), np.exp(ci[1])]
                print(f"    {var:25s}: OR={or_val:.3f} [{or_ci[0]:.3f},{or_ci[1]:.3f}], "
                      f"p={m2.pvalues[var]:.4e}")
        except Exception as e:
            print(f"    Failed: {e}")

        # Reg 3: DV = residual ΔP(irr) at D2
        df_corr = df_m.dropna(subset=['delta_p_irr_d2'])
        if len(df_corr) > 10:
            print(f"\n  --- Reg 3: DV = residual ΔP(irr) at D2, cluster-robust (N={len(df_corr)}) ---")
            try:
                m3 = smf.ols('delta_p_irr_d2 ~ distance + p_mode_baseline + committed_d1',
                               data=df_corr).fit(cov_type='cluster', cov_kwds={'groups': df_corr['qid']})
                print(f"  R² = {m3.rsquared:.4f}")
                for var in ['distance', 'p_mode_baseline', 'committed_d1']:
                    ci = m3.conf_int().loc[var]
                    print(f"    {var:25s}: β={m3.params[var]:+.4f}, t={m3.tvalues[var]:+.2f}, "
                          f"p={m3.pvalues[var]:.4e}, CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")
            except Exception as e:
                print(f"    Failed: {e}")

            # T-test
            comm_res = df_corr[df_corr['committed_d1'] == 1]['delta_p_irr_d2']
            uncomm_res = df_corr[df_corr['committed_d1'] == 0]['delta_p_irr_d2']
            if len(comm_res) > 1 and len(uncomm_res) > 1:
                t, p = ttest_ind(comm_res, uncomm_res, equal_var=False)
                pooled_sd = np.sqrt(((len(comm_res)-1)*comm_res.std()**2 +
                                     (len(uncomm_res)-1)*uncomm_res.std()**2) /
                                    (len(comm_res)+len(uncomm_res)-2))
                d_eff = (comm_res.mean() - uncomm_res.mean()) / pooled_sd if pooled_sd > 0 else 0
                print(f"\n  Committed vs Uncommitted residual:")
                print(f"    Committed:   M={comm_res.mean():+.3f} (SD={comm_res.std():.3f}, n={len(comm_res)})")
                print(f"    Uncommitted: M={uncomm_res.mean():+.3f} (SD={uncomm_res.std():.3f}, n={len(uncomm_res)})")
                print(f"    t={t:.3f}, p={p:.4e}, d={d_eff:.3f}")

        # Reg 4: DV = persisted (logistic, committed only, SECONDARY)
        committed_df = df_m[df_m['committed_d1'] == 1].copy()
        if 'persisted_d2' in committed_df.columns and len(committed_df) > 15:
            print(f"\n  --- Reg 4 [SECONDARY]: DV = persisted (logistic, committed only, N={len(committed_df)}) ---")
            try:
                m4 = smf.logit('persisted_d2 ~ distance + p_mode_baseline + d1_commitment_strength',
                                data=committed_df).fit(disp=0,
                                cov_type='cluster', cov_kwds={'groups': committed_df['qid']})
                print(f"  Pseudo R² = {m4.prsquared:.4f}")
                for var in ['distance', 'p_mode_baseline', 'd1_commitment_strength']:
                    or_val = np.exp(m4.params[var])
                    ci = m4.conf_int().loc[var]
                    or_ci = [np.exp(ci[0]), np.exp(ci[1])]
                    print(f"    {var:30s}: OR={or_val:.3f} [{or_ci[0]:.3f},{or_ci[1]:.3f}], "
                          f"p={m4.pvalues[var]:.4e}")
            except Exception as e:
                print(f"    Failed: {e}")

    except Exception as e:
        print(f"{model_name}: FAILED — {e}")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 6: Study 2A — Categorical framing, planned contrasts
# ══════════════════════════════════════════════════════════

print("="*70)
print("STUDY 2A — Categorical framing, planned contrasts, cluster-robust")
print("="*70)

for model_name in df_s2['model'].unique():
    df_m = df_s2[(df_s2['model'] == model_name) &
                  (df_s2['experiment'] == '2A') &
                  (df_s2['stage'] == 'commit')].copy()
    if len(df_m) < 20: continue
    n_dilemmas = df_m['qid'].nunique()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)}, {n_dilemmas} dilemmas)")
    print(f"{'━'*60}")

    # Rates with Wilson CIs
    print("\n  Commitment rates [95% Wilson CI]:")
    for f in ['memory', 'instruction', 'suggestion']:
        sub = df_m[df_m['framing'] == f]
        n = len(sub)
        k = int(sub['committed'].sum())
        rate = k / n * 100
        ci_lo, ci_hi = proportion_confint(k, n, method='wilson')
        print(f"    {f:15s}: {rate:.1f}% [{ci_lo*100:.1f}, {ci_hi*100:.1f}] (n={n})")

    # Planned contrasts
    print("\n  Planned contrasts (Bonferroni α=.017):")
    for f1, f2 in [('memory', 'suggestion'), ('memory', 'instruction'), ('instruction', 'suggestion')]:
        sub = df_m[df_m['framing'].isin([f1, f2])]
        ct = pd.crosstab(sub['framing'], sub['committed'])
        if ct.shape[1] >= 2:
            chi2, p, _, _ = chi2_contingency(ct)
            a = df_m[(df_m['framing']==f1) & (df_m['committed']==1)].shape[0]
            b = df_m[(df_m['framing']==f1) & (df_m['committed']==0)].shape[0]
            c = df_m[(df_m['framing']==f2) & (df_m['committed']==1)].shape[0]
            d = df_m[(df_m['framing']==f2) & (df_m['committed']==0)].shape[0]
            or_val = (a*d)/(b*c) if b*c > 0 else float('inf')
            sig = '***' if p < .001 else ('**' if p < .017 else 'ns')
            print(f"    {f1} vs {f2}: χ²={chi2:.2f}, p={p:.4e}, OR={or_val:.2f} {sig}")

    # Logistic with categorical framing
    print("\n  Logistic (ref=suggestion, cluster-robust):")
    df_m['framing_cat'] = pd.Categorical(df_m['framing'],
                                          categories=['suggestion', 'instruction', 'memory'])
    try:
        logit_m = smf.logit('committed ~ C(framing_cat) + distance + p_mode_baseline',
                             data=df_m).fit(disp=0,
                             cov_type='cluster', cov_kwds={'groups': df_m['qid']})
        print(f"  Pseudo R² = {logit_m.prsquared:.4f}")
        for var in logit_m.params.index:
            if var == 'Intercept': continue
            or_val = np.exp(logit_m.params[var])
            ci = logit_m.conf_int().loc[var]
            or_ci = [np.exp(ci[0]), np.exp(ci[1])]
            print(f"    {var:50s}: OR={or_val:.3f} [{or_ci[0]:.3f},{or_ci[1]:.3f}], p={logit_m.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Logistic failed: {e}")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 7: Study 2B — Categorical attribution, planned contrasts
# ══════════════════════════════════════════════════════════

print("="*70)
print("STUDY 2B — Categorical attribution, planned contrasts, cluster-robust")
print("="*70)

for model_name in df_s2['model'].unique():
    df_m = df_s2[(df_s2['model'] == model_name) &
                  (df_s2['experiment'] == '2B') &
                  (df_s2['stage'] == 'commit')].copy()
    if len(df_m) < 20: continue
    n_dilemmas = df_m['qid'].nunique()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)}, {n_dilemmas} dilemmas)")
    print(f"{'━'*60}")

    # Rates
    print("\n  Commitment rates [95% Wilson CI]:")
    for a in ['self', 'user', 'other_ai']:
        sub = df_m[df_m['attribution'] == a]
        n = len(sub)
        k = int(sub['committed'].sum())
        rate = k / n * 100
        ci_lo, ci_hi = proportion_confint(k, n, method='wilson')
        print(f"    {a:15s}: {rate:.1f}% [{ci_lo*100:.1f}, {ci_hi*100:.1f}] (n={n})")

    # Planned contrasts
    print("\n  Planned contrasts (Bonferroni α=.017):")
    for a1, a2 in [('self', 'user'), ('self', 'other_ai'), ('user', 'other_ai')]:
        sub = df_m[df_m['attribution'].isin([a1, a2])]
        ct = pd.crosstab(sub['attribution'], sub['committed'])
        if ct.shape[1] >= 2:
            chi2, p, _, _ = chi2_contingency(ct)
            ac = df_m[(df_m['attribution']==a1) & (df_m['committed']==1)].shape[0]
            bc = df_m[(df_m['attribution']==a1) & (df_m['committed']==0)].shape[0]
            cc = df_m[(df_m['attribution']==a2) & (df_m['committed']==1)].shape[0]
            dc = df_m[(df_m['attribution']==a2) & (df_m['committed']==0)].shape[0]
            or_val = (ac*dc)/(bc*cc) if bc*cc > 0 else float('inf')
            sig = '***' if p < .001 else ('**' if p < .017 else 'ns')
            r1 = df_m[df_m['attribution']==a1]['committed'].mean()*100
            r2 = df_m[df_m['attribution']==a2]['committed'].mean()*100
            print(f"    {a1} ({r1:.1f}%) vs {a2} ({r2:.1f}%): χ²={chi2:.2f}, p={p:.4e}, OR={or_val:.2f} {sig}")
        else:
            # Complete separation — use Fisher exact
            ct_full = pd.crosstab(sub['attribution'], sub['committed'])
            if ct_full.shape == (2, 2):
                odds, p = stats.fisher_exact(ct_full)
                print(f"    {a1} vs {a2}: Fisher exact OR={odds:.3f}, p={p:.4e}")
            else:
                print(f"    {a1} vs {a2}: cannot compute (no variation)")

    # Logistic
    print("\n  Logistic (ref=other_ai, cluster-robust):")
    df_m['attr_cat'] = pd.Categorical(df_m['attribution'],
                                       categories=['other_ai', 'user', 'self'])
    try:
        logit_m = smf.logit('committed ~ C(attr_cat) + distance + p_mode_baseline',
                             data=df_m).fit(disp=0,
                             cov_type='cluster', cov_kwds={'groups': df_m['qid']})
        print(f"  Pseudo R² = {logit_m.prsquared:.4f}")
        for var in logit_m.params.index:
            if var == 'Intercept': continue
            or_val = np.exp(logit_m.params[var])
            ci = logit_m.conf_int().loc[var]
            or_ci = [np.exp(ci[0]), np.exp(ci[1])]
            print(f"    {var:50s}: OR={or_val:.3f} [{or_ci[0]:.3f},{or_ci[1]:.3f}], p={logit_m.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Logistic failed (likely separation): {e}")
        print("    Using Fisher exact for pairwise instead (reported above).")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 8: Study 2 — Persistence (secondary, conditional)
# ══════════════════════════════════════════════════════════

print("="*70)
print("STUDY 2 — PERSISTENCE (secondary: conditional on committed)")
print("="*70)

for exp, groupvar in [('2A', 'framing'), ('2B', 'attribution')]:
    print(f"\n{'━'*40} Experiment {exp} {'━'*40}")
    levels = ['memory','instruction','suggestion'] if exp == '2A' else ['self','user','other_ai']

    for model_name in df_s2['model'].unique():
        df_m = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == exp)]
        commit_df = df_m[df_m['stage'] == 'commit']
        correct_df = df_m[df_m['stage'] == 'correct']
        if len(commit_df) < 10: continue

        print(f"\n  {model_name}:")
        persist_data = []
        for lev in levels:
            committed = commit_df[(commit_df[groupvar] == lev) & (commit_df['committed'] == 1)]
            keys = set(zip(committed['qid'], committed['irr']))
            corrections = correct_df[
                (correct_df[groupvar] == lev) &
                correct_df.apply(lambda x: (x['qid'], x['irr']) in keys, axis=1)
            ]
            n_c = len(committed)
            n_p = corrections['committed'].sum() if len(corrections) > 0 else 0
            rate = n_p / n_c * 100 if n_c > 0 else 0
            ci_lo, ci_hi = proportion_confint(int(n_p), n_c, method='wilson') if n_c > 0 else (0, 0)
            print(f"    {lev:15s}: {rate:.1f}% [{ci_lo*100:.1f},{ci_hi*100:.1f}] (n={n_c})")
            for _, row in corrections.iterrows():
                persist_data.append({groupvar: lev, 'persisted': row['committed']})

        pdf = pd.DataFrame(persist_data)
        if len(pdf) > 0:
            ct = pd.crosstab(pdf[groupvar], pdf['persisted'])
            if ct.shape[1] >= 2:
                chi2, p, dof, _ = chi2_contingency(ct)
                print(f"    χ²({dof})={chi2:.3f}, p={p:.4f}")
            else:
                print(f"    Cannot test (no variation in persistence)")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 9: Study 3 — Categorical ratio + interaction, cluster-robust
# ══════════════════════════════════════════════════════════
# NOTE: This cell assumes df_s3 already exists from your Study 3 analysis notebook.
# If not, load it here:
# df_s3 = pd.read_csv("path/to/study3_df.csv")
df_s3 = 
print("="*70)
print("STUDY 3 — Categorical ratio + distance interaction, cluster-robust")
print("="*70)

for model_name in df_s3['model'].unique():
    df_m = df_s3[df_s3['model'] == model_name].copy()
    df_m = df_m.dropna(subset=['delta_p_initial', 'p_mode_baseline', 'base_extremity'])
    n_dilemmas = df_m['qid'].nunique()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)}, {n_dilemmas} dilemmas)")
    print(f"{'━'*60}")

    df_m['ratio_cat'] = pd.Categorical(df_m['ratio'], categories=['0:3','1:2','2:1','3:0'])

    # Primary: categorical ratio
    print("\n  --- ΔP(initial) ~ ratio (categorical) + controls ---")
    try:
        m = smf.ols('delta_p_initial ~ C(ratio_cat) + p_mode_baseline + base_extremity',
                      data=df_m).fit(cov_type='cluster', cov_kwds={'groups': df_m['qid']})
        print(f"  R² = {m.rsquared:.4f}")
        for var in m.params.index:
            if var == 'Intercept': continue
            ci = m.conf_int().loc[var]
            print(f"    {var:45s}: β={m.params[var]:+.4f}, t={m.tvalues[var]:+.2f}, "
                  f"p={m.pvalues[var]:.4e}, CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")
    except Exception as e:
        print(f"    Failed: {e}")

    # Interaction: n_support × avg_dist_peers
    if 'avg_dist_peers' in df_m.columns:
        print("\n  --- + n_support × avg_dist_peers interaction ---")
        try:
            m2 = smf.ols('delta_p_initial ~ n_support * avg_dist_peers + p_mode_baseline + base_extremity',
                           data=df_m).fit(cov_type='cluster', cov_kwds={'groups': df_m['qid']})
            print(f"  R² = {m2.rsquared:.4f}")
            for var in m2.params.index:
                if var == 'Intercept': continue
                ci = m2.conf_int().loc[var]
                print(f"    {var:45s}: β={m2.params[var]:+.4f}, t={m2.tvalues[var]:+.2f}, "
                      f"p={m2.pvalues[var]:.4e}, CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")
        except Exception as e:
            print(f"    Failed: {e}")

print("\n\nAll corrected analyses complete.")

In [23]:
STUDY3_MODELS = {
    "GPT-4o": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_gpt.jsonl",
    "DeepSeek-V3.2": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/outputs/study3_deepseek/summaries_deepseek.jsonl",
    "Phi-4": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_phi.jsonl",
    "Qwen-2.5": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_qwen.jsonl",
}

def build_study3_df(path, model_name):
    with open(path) as f:
        rows = [json.loads(line) for line in f if line.strip()]

    records = []
    for r in rows:
        if r.get("final_choice") is None or r.get("final_dist_probs") is None:
            continue

        # Skip rows with None inside probability dicts
        if None in r["baseline_dist_probs"].values() or None in r["final_dist_probs"].values():
            continue
        base_dist = get_dist7(r["baseline_dist_probs"])
        final_dist = get_dist7(r["final_dist_probs"])
        bc, fc = r["base_choice"], r["final_choice"]

        records.append({
            "model": model_name,
            "qid": r["qid"],
            "condition": r["condition"],
            "ratio": r["ratio"],
            "base_choice": bc,
            "final_choice": fc,
            "base_extremity": abs(bc - 4),
            "p_mode_baseline": base_dist[bc - 1],
            "opp_extremity": parse_opp_extremity(r["condition"]),
            "n_opposers": sum(1 for rl in r["peer_roles"] if rl == "corrector"),
            # DVs
            "switched": int(fc != bc),
            "shift_magnitude": abs(fc - bc),
            "delta_p_initial": final_dist[bc - 1] - base_dist[bc - 1],
            "w1": wasserstein_1d(base_dist, final_dist),
        })
    return pd.DataFrame(records)

s3_frames = []
for name, path in STUDY3_MODELS.items():
    try:
        df = build_study3_df(path, name)
        s3_frames.append(df)
        print(f"{name}: {len(df)} trials, switched={df['switched'].mean()*100:.1f}%")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s3 = pd.concat(s3_frames, ignore_index=True)
df_s3['n_support'] = df_s3['ratio'].map({'3:0': 3, '2:1': 2, '1:2': 1, '0:3': 0})

print(f"\nTotal: {len(df_s3)} trials, {df_s3['model'].nunique()} model(s)")
print(df_s3.groupby(['model', 'ratio'])['switched'].agg(['mean', 'count']).round(3))

GPT-4o: 5692 trials, switched=22.6%
DeepSeek-V3.2: 5508 trials, switched=31.2%
Phi-4: 5694 trials, switched=53.0%
Qwen-2.5: 5694 trials, switched=61.8%

Total: 22588 trials, 4 model(s)
                      mean  count
model         ratio              
DeepSeek-V3.2 0:3    0.523    682
              1:2    0.317   2702
              2:1    0.238   1824
              3:0    0.247    300
GPT-4o        0:3    0.807    701
              1:2    0.119   2807
              2:1    0.173   1872
              3:0    0.212    312
Phi-4         0:3    0.893    702
              1:2    0.600   2808
              2:1    0.338   1872
              3:0    0.244    312
Qwen-2.5      0:3    0.749    702
              1:2    0.668   2808
              2:1    0.544   1872
              3:0    0.321    312


In [14]:
print("Columns in df_s3:")
print(list(df_s3.columns))
print(f"\nShape: {df_s3.shape}")
print(f"\nSample row:")
print(df_s3.iloc[0].to_dict())

Columns in df_s3:
['model', 'qid', 'condition', 'ratio', 'base_choice', 'final_choice', 'base_extremity', 'p_mode_baseline', 'opp_extremity', 'n_opposers', 'switched', 'shift_magnitude', 'delta_p_initial', 'w1', 'n_support']

Shape: (22588, 15)

Sample row:
{'model': 'GPT-4o', 'qid': 'moral_1', 'condition': 'sup3_match', 'ratio': '3:0', 'base_choice': 6, 'final_choice': 6, 'base_extremity': 2, 'p_mode_baseline': 0.13246572451000402, 'opp_extremity': None, 'n_opposers': 0, 'switched': 0, 'shift_magnitude': 0, 'delta_p_initial': 0.8675341620686394, 'w1': 0.8709631810450783, 'n_support': 3}


In [15]:
print("Unique conditions (sample):")
print(df_s3['condition'].unique()[:30])
print(f"\nTotal unique conditions: {df_s3['condition'].nunique()}")

Unique conditions (sample):
['sup3_match' 'sup3_close' 'sup3_moderate' 'sup3_extreme' 'opp3_mild_n4L'
 'opp3_mild_n4R' 'opp3_mild_n4Split' 'opp3_moderate_n4L'
 'opp3_moderate_n4R' 'opp3_moderate_n4Split' 'opp3_extreme_n4L'
 'opp3_extreme_n4R' 'opp3_extreme_n4Split' 'sup2_match_opp1_mild_n4L'
 'sup2_match_opp1_mild_n4R' 'sup2_match_opp1_moderate_n4L'
 'sup2_match_opp1_moderate_n4R' 'sup2_match_opp1_extreme_n4L'
 'sup2_match_opp1_extreme_n4R' 'sup2_close_opp1_mild_n4L'
 'sup2_close_opp1_mild_n4R' 'sup2_close_opp1_moderate_n4L'
 'sup2_close_opp1_moderate_n4R' 'sup2_close_opp1_extreme_n4L'
 'sup2_close_opp1_extreme_n4R' 'sup2_moderate_opp1_mild_n4L'
 'sup2_moderate_opp1_mild_n4R' 'sup2_moderate_opp1_moderate_n4L'
 'sup2_moderate_opp1_moderate_n4R' 'sup2_moderate_opp1_extreme_n4L']

Total unique conditions: 73


In [24]:
def parse_condition_distance(row):
    """Extract average peer distance from condition string."""
    cond = str(row['condition'])
    base = row['base_choice']
    
    # Distance labels → approximate Likert distances
    dist_map = {'match': 0, 'close': 1, 'moderate': 2, 'extreme': 3, 'mild': 1.5}
    
    # Parse: sup3_match, opp3_mild_n4L, sup2_match_opp1_mild_n4L, etc.
    # Remove the n4L/n4R/n4Split suffix
    clean = cond
    for suffix in ['_n4L', '_n4R', '_n4Split']:
        clean = clean.replace(suffix, '')
    
    parts = clean.split('_')
    distances = []
    
    i = 0
    while i < len(parts):
        p = parts[i]
        if p.startswith('sup'):
            n = int(p[3:]) if len(p) > 3 and p[3:].isdigit() else 1
            if i + 1 < len(parts) and parts[i+1] in dist_map:
                d = dist_map[parts[i+1]]
                distances.extend([d] * n)
                i += 2
            else:
                i += 1
        elif p.startswith('opp'):
            n = int(p[3:]) if len(p) > 3 and p[3:].isdigit() else 1
            if i + 1 < len(parts) and parts[i+1] in dist_map:
                d = dist_map[parts[i+1]]
                distances.extend([d] * n)
                i += 2
            else:
                i += 1
        else:
            i += 1
    
    if distances:
        return pd.Series({
            'avg_dist_peers': np.mean(distances),
            'max_dist_peers': np.max(distances),
            'peer_spread': np.std(distances) if len(distances) > 1 else 0,
            'n_peers': len(distances),
        })
    else:
        return pd.Series({
            'avg_dist_peers': np.nan,
            'max_dist_peers': np.nan,
            'peer_spread': np.nan,
            'n_peers': 0,
        })

# Compute
print("Parsing conditions...")
dist_df = df_s3.apply(parse_condition_distance, axis=1)
df_s3 = pd.concat([df_s3, dist_df], axis=1)

# Verify
print(f"\navg_dist_peers stats:")
print(df_s3['avg_dist_peers'].describe())
print(f"\nVerification — parsed conditions:")
for cond in ['sup3_match', 'sup3_extreme', 'opp3_mild_n4L', 'opp3_extreme_n4R',
             'sup2_match_opp1_mild_n4L', 'sup2_moderate_opp1_extreme_n4R',
             'sup1_close_opp2_moderate_n4L']:
    subset = df_s3[df_s3['condition'] == cond]
    if len(subset) > 0:
        r = subset.iloc[0]
        print(f"  {cond:45s} → avg_dist={r['avg_dist_peers']:.2f}, "
              f"spread={r['peer_spread']:.2f}, n_peers={int(r['n_peers'])}")

Parsing conditions...

avg_dist_peers stats:
count    22588.000000
mean         1.874314
std          0.703942
min          0.000000
25%          1.333333
50%          2.000000
75%          2.333333
max          3.000000
Name: avg_dist_peers, dtype: float64

Verification — parsed conditions:
  sup3_match                                    → avg_dist=0.00, spread=0.00, n_peers=3
  sup3_extreme                                  → avg_dist=3.00, spread=0.00, n_peers=3
  opp3_mild_n4L                                 → avg_dist=1.50, spread=0.00, n_peers=3
  opp3_extreme_n4R                              → avg_dist=3.00, spread=0.00, n_peers=3
  sup2_match_opp1_mild_n4L                      → avg_dist=0.50, spread=0.71, n_peers=3
  sup2_moderate_opp1_extreme_n4R                → avg_dist=2.33, spread=0.47, n_peers=3
  sup1_close_opp2_moderate_n4L                  → avg_dist=1.67, spread=0.47, n_peers=3


In [18]:
from scipy.stats import f as f_dist
df_s3_clean = df_s3.dropna(subset=['avg_dist_peers', 'delta_p_initial', 
                                    'n_support', 'p_mode_baseline', 'base_extremity'])
print(f"Clean N: {len(df_s3_clean)}")

for model_name in df_s3_clean['model'].unique():
    df_m = df_s3_clean[df_s3_clean['model'] == model_name].copy()
    
    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)})")
    print(f"{'━'*60}")
    
    # ── Model 1: Main effects only ──
    print("\n  --- Model 1: Main effects ---")
    m1 = smf.ols('delta_p_initial ~ n_support + avg_dist_peers + p_mode_baseline + base_extremity',
                  data=df_m).fit()
    print(f"  R² = {m1.rsquared:.4f}")
    for var in ['n_support', 'avg_dist_peers', 'p_mode_baseline', 'base_extremity']:
        print(f"    {var:25s}: β={m1.params[var]:+.4f}, t={m1.tvalues[var]:+.2f}, p={m1.pvalues[var]:.4e}")
    
    # ── Model 2: + Coalition × Avg Distance interaction ──
    print("\n  --- Model 2: + n_support × avg_dist_peers ---")
    m2 = smf.ols('delta_p_initial ~ n_support * avg_dist_peers + p_mode_baseline + base_extremity',
                  data=df_m).fit()
    print(f"  R² = {m2.rsquared:.4f}")
    for var in m2.params.index:
        if var == 'Intercept': continue
        print(f"    {var:40s}: β={m2.params[var]:+.4f}, t={m2.tvalues[var]:+.2f}, p={m2.pvalues[var]:.4e}")
    
    # F-test: does interaction improve fit?
    ssr1, ssr2 = m1.ssr, m2.ssr
    df_diff = m2.df_model - m1.df_model
    f_stat = ((ssr1 - ssr2) / df_diff) / (ssr2 / m2.df_resid)
    p_val = 1 - f_dist.cdf(f_stat, df_diff, m2.df_resid)
    print(f"\n  Interaction improvement: F({df_diff},{m2.df_resid:.0f})={f_stat:.3f}, p={p_val:.4e}, ΔR²={m2.rsquared - m1.rsquared:.4f}")
    
    # ── Model 3: + Peer spread interaction ──
    if df_m['peer_spread'].std() > 0:
        print("\n  --- Model 3: + n_support × peer_spread ---")
        m3 = smf.ols('delta_p_initial ~ n_support * avg_dist_peers + n_support * peer_spread + p_mode_baseline + base_extremity',
                      data=df_m).fit()
        print(f"  R² = {m3.rsquared:.4f}")
        for var in m3.params.index:
            if var == 'Intercept': continue
            print(f"    {var:40s}: β={m3.params[var]:+.4f}, t={m3.tvalues[var]:+.2f}, p={m3.pvalues[var]:.4e}")
    
    # ── Model 4: Logistic — switching ──
    print("\n  --- Model 4: Logistic — switched ~ n_support × avg_dist ---")
    try:
        m4 = smf.logit('switched ~ n_support * avg_dist_peers + p_mode_baseline + base_extremity',
                        data=df_m).fit(disp=0)
        print(f"  Pseudo R² = {m4.prsquared:.4f}")
        for var in m4.params.index:
            if var == 'Intercept': continue
            or_val = np.exp(m4.params[var])
            print(f"    {var:40s}: β={m4.params[var]:+.4f}, OR={or_val:.3f}, p={m4.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Failed: {e}")
    
    # ── Simple slopes ──
    if 'n_support:avg_dist_peers' in m2.params:
        print("\n  --- Simple slopes: n_support effect at different distances ---")
        b_sup = m2.params['n_support']
        b_int = m2.params['n_support:avg_dist_peers']
        mean_d = df_m['avg_dist_peers'].mean()
        sd_d = df_m['avg_dist_peers'].std()
        for label, val in [('close (−1SD)', mean_d - sd_d), ('mean', mean_d), ('far (+1SD)', mean_d + sd_d)]:
            slope = b_sup + b_int * val
            print(f"    At dist={val:.2f} ({label}): support effect = {slope:+.4f}")
    
    # ── Descriptive crosstab ──
    print("\n  --- ΔP(initial) by ratio × distance tercile ---")
    df_m['dist_tercile'] = pd.qcut(df_m['avg_dist_peers'], 3, labels=['close', 'medium', 'far'], duplicates='drop')
    ct = df_m.groupby(['ratio', 'dist_tercile'])['delta_p_initial'].mean().unstack('dist_tercile')
    print(ct.to_string(float_format=lambda x: f"{x:+.3f}"))

Clean N: 22588

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  GPT-4o (N=5692)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  --- Model 1: Main effects ---
  R² = 0.6127
    n_support                : β=+0.0518, t=+10.36, p=6.0603e-25
    avg_dist_peers           : β=-0.0912, t=-16.98, p=3.7471e-63
    p_mode_baseline          : β=-0.9996, t=-89.33, p=0.0000e+00
    base_extremity           : β=+0.1063, t=+23.28, p=1.4669e-114

  --- Model 2: + n_support × avg_dist_peers ---
  R² = 0.6178
    n_support                               : β=+0.1514, t=+12.13, p=1.8100e-33
    avg_dist_peers                          : β=-0.0072, t=-0.66, p=5.1096e-01
    n_support:avg_dist_peers                : β=-0.0521, t=-8.70, p=4.2405e-18
    p_mode_baseline                         : β=-0.9992, t=-89.88, p=0.0000e+00
    base_extremity                          : β=+0.1064, t=+23.47, p=2.4511e-116

  Interaction improvement: F(1.0,5686)=75.712, p=1.1102e-16, ΔR²=0.0051



In [27]:

MODELS = {
    "GPT-4o": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/baseline_outputs_explicit_format.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/study1_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/study2_v3_experimental_outputs.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/gpt4o_study2_additional_results.jsonl",
    },
    "DeepSeek-V3.2": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/outputs/deepseek_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/outputs/deepseek_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/deepseek_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/outputs/deepseek_study2_additional_results.jsonl",
    },

    "Olmo-3.5": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/olmo_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/olmo_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/olmo_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/olmo_study2_additional_results.jsonl",
    },

    "qwen": {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/qwen_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/qwen_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/qwen_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/qwen_study2_additional_results.jsonl",

    },

    'Phi-4': {
        "baseline": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/phi4_study1_baseline_results.jsonl",
        "study1_full": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study1/phi4_study1_full_results.jsonl",
        "study2_testing": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/phi4_study2_testing_results.jsonl",
        "study2_additional": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study2/phi4_study2_additional_results.jsonl",

}
}

# Study 3 path (if separate)
STUDY3_MODELS = {
    "GPT-4o": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_gpt.jsonl",
    "DeepSeek-V3.2": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/outputs/study3_deepseek/summaries_deepseek.jsonl",
    "Phi-4": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_phi.jsonl",
    "Qwen-2.5": "/Users/wangbaihui/modeling_distributions_thesis/src/exp1_v2/outputs/exp1/study3/full_run/summaries_qwen.jsonl",
} # or build from raw

print("Paths defined. Update before running.")

Paths defined. Update before running.


In [28]:
# ══════════════════════════════════════════════════════════
# CELL 3: Build Study 1 DataFrame
# ══════════════════════════════════════════════════════════

def load_baseline(path):
    bl = {}
    for obj in load_jsonl(path):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        md = obj.get("metadata", {})
        qid = md.get("qid") or obj.get("custom_id", "").split("|")[0]
        probs = resp.get("probs", {})
        if qid and probs:
            dist = get_dist7(probs)
            mode = int(np.argmax(dist)) + 1
            bl[qid] = {"dist": dist, "mode": mode, "p_mode": dist[mode-1]}
    return bl

def build_study1_df(model_name, paths):
    bl = load_baseline(paths["baseline"])
    rows = []
    for obj in load_jsonl(paths["study1_full"]):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        probs = resp.get("probs", {})
        if not probs: continue
        md = obj.get("metadata", {})
        qid = md.get("qid") or obj.get("custom_id", "").split("|")[0]
        if qid not in bl: continue

        post_dist = get_dist7(probs)
        base = bl[qid]
        mode = base["mode"]

        cue_target = md.get("cue_target")
        if cue_target is None or cue_target == "none": continue
        cue_target = int(cue_target)

        cue_type = str(md.get("cue_direction", md.get("condition", ""))).lower()
        if "oppose" in cue_type: cue_type = "oppose"
        elif "pull" in cue_type: cue_type = "pull"
        elif "reinforce" in cue_type: cue_type = "reinforce"
        elif "memory" in cue_type: cue_type = "memory_only"

        distance = md.get("cue_distance")
        if distance is not None and distance != "none":
            distance = int(distance)
        else:
            distance = abs(cue_target - mode)

        rows.append({
            "model": model_name, "qid": qid, "mode": mode,
            "p_mode_baseline": base["p_mode"],
            "extremity": abs(mode - 4),
            "cue_target": cue_target, "cue_type": cue_type,
            "distance": distance,
            "dp_prior": post_dist[mode-1] - base["dist"][mode-1],
            "dp_cue": post_dist[cue_target-1] - base["dist"][cue_target-1],
            "w1": wasserstein_1d(base["dist"], post_dist),
        })
    return pd.DataFrame(rows)

# Build
s1_frames = []
for name, paths in MODELS.items():
    try:
        df = build_study1_df(name, paths)
        s1_frames.append(df)
        print(f"{name}: {len(df)} trials, {df[df['cue_type']=='oppose'].shape[0]} oppose")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s1 = pd.concat(s1_frames, ignore_index=True)
print(f"\nTotal Study 1: {len(df_s1)}")
print(df_s1.groupby(['model', 'cue_type']).size().unstack(fill_value=0))

GPT-4o: 464 trials, 220 oppose
DeepSeek-V3.2: 468 trials, 260 oppose
Olmo-3.5: 468 trials, 312 oppose
qwen: 468 trials, 260 oppose
Phi-4: 468 trials, 268 oppose

Total Study 1: 2336
cue_type       oppose  pull  reinforce
model                                 
DeepSeek-V3.2     260    78        130
GPT-4o            220   132        112
Olmo-3.5          312     0        156
Phi-4             268    66        134
qwen              260    78        130


In [29]:
# ══════════════════════════════════════════════════════════
# CELL 4: Build Study 2 DataFrame
# ══════════════════════════════════════════════════════════

def build_study2_df(model_name, paths):
    bl = load_baseline(paths["baseline"])
    rows = []
    for obj in load_jsonl(paths["study2_additional"]):
        resp = obj.get("response", {})
        if resp.get("status_code") != 200: continue
        probs = resp.get("probs", {})
        if not probs: continue
        md = obj.get("metadata", {})
        qid = md.get("qid")
        if qid not in bl: continue

        post_dist = get_dist7(probs)
        irr = md["irrational_choice"]
        mode = md["mode_choice"]

        rows.append({
            "model": model_name, "qid": qid,
            "experiment": md.get("experiment"),
            "framing": md.get("framing"),
            "attribution": md.get("attribution"),
            "stage": md.get("stage"),
            "irr": irr, "mode": mode,
            "distance": md.get("distance"),
            "p_mode_baseline": bl[qid]["p_mode"],
            "p_irr": post_dist[irr-1],
            "p_mode": post_dist[mode-1],
            "committed": int(post_dist[irr-1] > post_dist[mode-1]),
            "delta_p_irr": post_dist[irr-1] - bl[qid]["dist"][irr-1],
            "w1": wasserstein_1d(bl[qid]["dist"], post_dist),
        })
    return pd.DataFrame(rows)

s2_frames = []
for name, paths in MODELS.items():
    try:
        df = build_study2_df(name, paths)
        s2_frames.append(df)
        print(f"{name}: {len(df)} trials")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s2 = pd.concat(s2_frames, ignore_index=True)
print(f"\nTotal Study 2: {len(df_s2)}")

GPT-4o: 1740 trials
DeepSeek-V3.2: 5088 trials
Olmo-3.5: 5616 trials
qwen: 3528 trials
Phi-4: 2460 trials

Total Study 2: 18432


In [30]:
def build_study3_df(path, model_name):
    with open(path) as f:
        rows = [json.loads(line) for line in f if line.strip()]

    records = []
    for r in rows:
        if r.get("final_choice") is None or r.get("final_dist_probs") is None:
            continue

        # Skip rows with None inside probability dicts
        if None in r["baseline_dist_probs"].values() or None in r["final_dist_probs"].values():
            continue
        base_dist = get_dist7(r["baseline_dist_probs"])
        final_dist = get_dist7(r["final_dist_probs"])
        bc, fc = r["base_choice"], r["final_choice"]

        records.append({
            "model": model_name,
            "qid": r["qid"],
            "condition": r["condition"],
            "ratio": r["ratio"],
            "base_choice": bc,
            "final_choice": fc,
            "base_extremity": abs(bc - 4),
            "p_mode_baseline": base_dist[bc - 1],
            "opp_extremity": parse_opp_extremity(r["condition"]),
            "n_opposers": sum(1 for rl in r["peer_roles"] if rl == "corrector"),
            # DVs
            "switched": int(fc != bc),
            "shift_magnitude": abs(fc - bc),
            "delta_p_initial": final_dist[bc - 1] - base_dist[bc - 1],
            "w1": wasserstein_1d(base_dist, final_dist),
        })
    return pd.DataFrame(records)

s3_frames = []
for name, path in STUDY3_MODELS.items():
    try:
        df = build_study3_df(path, name)
        s3_frames.append(df)
        print(f"{name}: {len(df)} trials, switched={df['switched'].mean()*100:.1f}%")
    except Exception as e:
        print(f"{name}: FAILED — {e}")

df_s3 = pd.concat(s3_frames, ignore_index=True)
df_s3['n_support'] = df_s3['ratio'].map({'3:0': 3, '2:1': 2, '1:2': 1, '0:3': 0})

print(f"\nTotal: {len(df_s3)} trials, {df_s3['model'].nunique()} model(s)")
print(df_s3.groupby(['model', 'ratio'])['switched'].agg(['mean', 'count']).round(3))

GPT-4o: 5692 trials, switched=22.6%
DeepSeek-V3.2: 5508 trials, switched=31.2%
Phi-4: 5694 trials, switched=53.0%
Qwen-2.5: 5694 trials, switched=61.8%

Total: 22588 trials, 4 model(s)
                      mean  count
model         ratio              
DeepSeek-V3.2 0:3    0.523    682
              1:2    0.317   2702
              2:1    0.238   1824
              3:0    0.247    300
GPT-4o        0:3    0.807    701
              1:2    0.119   2807
              2:1    0.173   1872
              3:0    0.212    312
Phi-4         0:3    0.893    702
              1:2    0.600   2808
              2:1    0.338   1872
              3:0    0.244    312
Qwen-2.5      0:3    0.749    702
              1:2    0.668   2808
              2:1    0.544   1872
              3:0    0.321    312


In [31]:
# ══════════════════════════════════════════════════════════
# CELL 5: Study 1 — Corrected regressions (mixed-effects or cluster-robust)
# ══════════════════════════════════════════════════════════

print("="*70)
print("STUDY 1 — Primary: ΔP(cue target) ~ distance + distance² + controls")
print("Oppose trials only. Cluster-robust SEs by dilemma.")
print("="*70)

for model_name in df_s1['model'].unique():
    opp = df_s1[(df_s1['model'] == model_name) & (df_s1['cue_type'] == 'oppose')].copy()
    opp = opp.dropna(subset=['dp_cue', 'distance', 'p_mode_baseline'])
    opp['distance_sq'] = opp['distance'] ** 2
    n_dilemmas = opp['qid'].nunique()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(opp)}, {n_dilemmas} dilemmas)")
    print(f"{'━'*60}")

    use_mixed = HAS_MIXED
    if use_mixed:
        try:
            m_lin = smf.mixedlm('dp_cue ~ distance + p_mode_baseline + extremity',
                                 data=opp, groups=opp['qid']).fit(reml=True)
            m_quad = smf.mixedlm('dp_cue ~ distance + distance_sq + p_mode_baseline + extremity',
                                  data=opp, groups=opp['qid']).fit(reml=True)

            print("\n  --- Linear (mixed-effects, random intercept for dilemma) ---")
            print(f"  Log-lik: {m_lin.llf:.2f}")
            for var in ['distance', 'p_mode_baseline', 'extremity']:
                ci = m_lin.conf_int().loc[var]
                print(f"    {var:20s}: β={m_lin.fe_params[var]:+.4f}, "
                      f"SE={m_lin.bse_fe[var]:.4f}, z={m_lin.tvalues[var]:+.2f}, "
                      f"p={m_lin.pvalues[var]:.4e}, 95%CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")

            print(f"\n  --- Quadratic (mixed-effects) ---")
            print(f"  Log-lik: {m_quad.llf:.2f}")
            for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
                ci = m_quad.conf_int().loc[var]
                print(f"    {var:20s}: β={m_quad.fe_params[var]:+.4f}, "
                      f"SE={m_quad.bse_fe[var]:.4f}, z={m_quad.tvalues[var]:+.2f}, "
                      f"p={m_quad.pvalues[var]:.4e}, 95%CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")

            lr_stat = 2 * (m_quad.llf - m_lin.llf)
            lr_p = 1 - stats.chi2.cdf(lr_stat, df=1)
            print(f"\n  LR test (linear vs quadratic): χ²(1)={lr_stat:.3f}, p={lr_p:.4e}")

        except Exception as e:
            print(f"  Mixed-effects failed: {e}")
            print(f"  Falling back to cluster-robust SEs.")
            use_mixed = False

    if not use_mixed:
        m_lin = smf.ols('dp_cue ~ distance + p_mode_baseline + extremity',
                          data=opp).fit(cov_type='cluster', cov_kwds={'groups': opp['qid']})
        m_quad = smf.ols('dp_cue ~ distance + distance_sq + p_mode_baseline + extremity',
                           data=opp).fit(cov_type='cluster', cov_kwds={'groups': opp['qid']})

        print(f"\n  --- Linear (OLS, cluster-robust SEs) ---")
        print(f"  R² = {m_lin.rsquared:.4f}")
        for var in ['distance', 'p_mode_baseline', 'extremity']:
            ci = m_lin.conf_int().loc[var]
            print(f"    {var:20s}: β={m_lin.params[var]:+.4f}, SE={m_lin.bse[var]:.4f}, "
                  f"t={m_lin.tvalues[var]:+.2f}, p={m_lin.pvalues[var]:.4e}, "
                  f"95%CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")

        print(f"\n  --- Quadratic (OLS, cluster-robust SEs) ---")
        print(f"  R² = {m_quad.rsquared:.4f}")
        for var in ['distance', 'distance_sq', 'p_mode_baseline', 'extremity']:
            ci = m_quad.conf_int().loc[var]
            print(f"    {var:20s}: β={m_quad.params[var]:+.4f}, SE={m_quad.bse[var]:.4f}, "
                  f"t={m_quad.tvalues[var]:+.2f}, p={m_quad.pvalues[var]:.4e}, "
                  f"95%CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")

        from scipy.stats import f as f_dist
        ssr1, ssr2 = m_lin.ssr, m_quad.ssr
        f_stat = ((ssr1 - ssr2) / 1) / (ssr2 / m_quad.df_resid)
        f_p = 1 - f_dist.cdf(f_stat, 1, m_quad.df_resid)
        print(f"\n  Quadratic improvement: F(1,{m_quad.df_resid:.0f})={f_stat:.3f}, "
              f"p={f_p:.4e}, ΔR²={m_quad.rsquared - m_lin.rsquared:.4f}")

STUDY 1 — Primary: ΔP(cue target) ~ distance + distance² + controls
Oppose trials only. Cluster-robust SEs by dilemma.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  GPT-4o (N=220, 55 dilemmas)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  --- Linear (mixed-effects, random intercept for dilemma) ---
  Log-lik: 118.03
    distance            : β=-0.0292, SE=0.0081, z=-3.61, p=3.0788e-04, 95%CI=[-0.0450,-0.0133]
    p_mode_baseline     : β=-0.0974, SE=0.0615, z=-1.58, p=1.1365e-01, 95%CI=[-0.2180,+0.0233]
    extremity           : β=+0.0119, SE=0.0179, z=+0.66, p=5.0626e-01, 95%CI=[-0.0231,+0.0469]

  --- Quadratic (mixed-effects) ---
  Log-lik: 117.31
    distance            : β=-0.1333, SE=0.0390, z=-3.41, p=6.4076e-04, 95%CI=[-0.2098,-0.0567]
    distance_sq         : β=+0.0128, SE=0.0047, z=+2.72, p=6.4563e-03, 95%CI=[+0.0036,+0.0220]
    p_mode_baseline     : β=-0.1070, SE=0.0602, z=-1.78, p=7.5600e-02, 95%CI=[-0.2250,+0.0110]
    extremity        

In [32]:
print("="*70)
print("STUDY 2 ORIGINAL — Memory injection → Correction")
print("="*70)

for model_name, paths in MODELS.items():
    try:
        path = paths["study2_testing"]
        df_m = build_study2_orig(path, model_name)

def build_study2_orig(path, model_name):
    rows_raw = load_jsonl(path)
    bl = load_baseline(MODELS[model_name]["baseline"])

    records = {}
    for r in rows_raw:
        resp = r.get("response", {})
        if resp.get("status_code") != 200: continue
        md = r.get("metadata", {})
        qid = md.get("qid")
        irr = md.get("irrational_choice")
        condition = md.get("condition", "")
        if not qid or irr is None or qid not in bl: continue
        probs = resp.get("probs", {})
        if not probs: continue

        dist = get_dist7(probs)
        key = (qid, irr)
        if key not in records:
            records[key] = {
                "qid": qid, "irr": irr,
                "mode": bl[qid]["mode"],
                "distance": md.get("distance"),
                "p_mode_baseline": bl[qid]["p_mode"],
                "p_irr_baseline": bl[qid]["dist"][irr - 1],
            }
        records[key][f"{condition}_dist"] = dist

    rows_out = []
    for key, rec in records.items():
        if "memory_justify_dist" not in rec: continue
        d1 = rec["memory_justify_dist"]
        irr, mode = rec["irr"], rec["mode"]
        p_irr_bl = rec["p_irr_baseline"]
        p_mode_bl = rec["p_mode_baseline"]

        d1_p_irr = d1[irr - 1]
        d1_p_mode = d1[mode - 1]
        committed = d1_p_irr > d1_p_mode

        row = {
            "model": model_name, "qid": rec["qid"],
            "irr": irr, "mode": mode,
            "distance": rec.get("distance"),
            "p_mode_baseline": p_mode_bl,
            "p_irr_baseline": p_irr_bl,
            "committed_d1": int(committed),
            "delta_p_irr_d1": d1_p_irr - p_irr_bl,
            "d1_p_irr": d1_p_irr,
            "d1_commitment_strength": d1_p_irr - d1_p_mode,
        }

        if "memory_justify_correction_dist" in rec:
            d2 = rec["memory_justify_correction_dist"]
            d2_p_irr = d2[irr - 1]
            d2_p_mode = d2[mode - 1]
            row["persisted_d2"] = int(d2_p_irr > d2_p_mode)
            row["delta_p_irr_d2"] = d2_p_irr - p_irr_bl
            row["w1_d1_d2"] = wasserstein_1d(d1, d2)

        rows_out.append(row)
    return pd.DataFrame(rows_out)

# Run for all models
for model_name, paths in MODELS.items():
    try:
        df_m = build_study2_orig(paths["study2_testing"], model_name)
        df_m['distance'] = pd.to_numeric(df_m['distance'], errors='coerce')
        df_m['p_mode_baseline'] = pd.to_numeric(df_m['p_mode_baseline'], errors='coerce')
        df_m['p_irr_baseline'] = pd.to_numeric(df_m['p_irr_baseline'], errors='coerce')
        df_m = df_m.dropna(subset=['distance', 'p_mode_baseline'])
        n_dilemmas = df_m['qid'].nunique()

        print(f"\n{'━'*60}")
        print(f"  {model_name} (N={len(df_m)}, {n_dilemmas} dilemmas)")
        print(f"{'━'*60}")

        # Descriptives
        print(f"\n  Descriptives:")
        print(f"    Commitment rate: {df_m['committed_d1'].mean()*100:.1f}%")
        ci_lo, ci_hi = proportion_confint(int(df_m['committed_d1'].sum()), len(df_m), method='wilson')
        print(f"    95% CI: [{ci_lo*100:.1f}, {ci_hi*100:.1f}]")
        print(f"    Mean ΔP(irr) at D1: {df_m['delta_p_irr_d1'].mean():+.3f}")
        if 'persisted_d2' in df_m.columns:
            comm = df_m[df_m['committed_d1'] == 1]
            if len(comm) > 0:
                pers_rate = comm['persisted_d2'].mean() * 100
                ci_lo2, ci_hi2 = proportion_confint(int(comm['persisted_d2'].sum()), len(comm), method='wilson')
                print(f"    Persistence: {pers_rate:.1f}% [{ci_lo2*100:.1f}, {ci_hi2*100:.1f}] "
                      f"({int(comm['persisted_d2'].sum())}/{len(comm)})")
        if 'delta_p_irr_d2' in df_m.columns:
            print(f"    Mean residual ΔP(irr) at D2: {df_m['delta_p_irr_d2'].mean():+.3f}")

        # Reg 1: DV = ΔP(irr) at D1
        print(f"\n  --- Reg 1: DV = ΔP(irr) at D1, cluster-robust ---")
        try:
            m1 = smf.ols('delta_p_irr_d1 ~ distance + p_mode_baseline + p_irr_baseline',
                           data=df_m).fit(cov_type='cluster', cov_kwds={'groups': df_m['qid']})
            print(f"  R² = {m1.rsquared:.4f}")
            for var in ['distance', 'p_mode_baseline', 'p_irr_baseline']:
                ci = m1.conf_int().loc[var]
                print(f"    {var:25s}: β={m1.params[var]:+.4f}, t={m1.tvalues[var]:+.2f}, "
                      f"p={m1.pvalues[var]:.4e}, CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")
        except Exception as e:
            print(f"    Failed: {e}")

        # Reg 2: DV = committed (logistic)
        print(f"\n  --- Reg 2: DV = committed_d1 (logistic, cluster-robust) ---")
        try:
            m2 = smf.logit('committed_d1 ~ distance + p_mode_baseline + p_irr_baseline',
                             data=df_m).fit(disp=0,
                             cov_type='cluster', cov_kwds={'groups': df_m['qid']})
            print(f"  Pseudo R² = {m2.prsquared:.4f}")
            for var in ['distance', 'p_mode_baseline', 'p_irr_baseline']:
                or_val = np.exp(m2.params[var])
                ci = m2.conf_int().loc[var]
                or_ci = [np.exp(ci[0]), np.exp(ci[1])]
                print(f"    {var:25s}: OR={or_val:.3f} [{or_ci[0]:.3f},{or_ci[1]:.3f}], "
                      f"p={m2.pvalues[var]:.4e}")
        except Exception as e:
            print(f"    Failed: {e}")

        # Reg 3: DV = residual ΔP(irr) at D2
        df_corr = df_m.dropna(subset=['delta_p_irr_d2'])
        if len(df_corr) > 10:
            print(f"\n  --- Reg 3: DV = residual ΔP(irr) at D2, cluster-robust (N={len(df_corr)}) ---")
            try:
                m3 = smf.ols('delta_p_irr_d2 ~ distance + p_mode_baseline + committed_d1',
                               data=df_corr).fit(cov_type='cluster', cov_kwds={'groups': df_corr['qid']})
                print(f"  R² = {m3.rsquared:.4f}")
                for var in ['distance', 'p_mode_baseline', 'committed_d1']:
                    ci = m3.conf_int().loc[var]
                    print(f"    {var:25s}: β={m3.params[var]:+.4f}, t={m3.tvalues[var]:+.2f}, "
                          f"p={m3.pvalues[var]:.4e}, CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")
            except Exception as e:
                print(f"    Failed: {e}")

            # T-test
            comm_res = df_corr[df_corr['committed_d1'] == 1]['delta_p_irr_d2']
            uncomm_res = df_corr[df_corr['committed_d1'] == 0]['delta_p_irr_d2']
            if len(comm_res) > 1 and len(uncomm_res) > 1:
                t, p = ttest_ind(comm_res, uncomm_res, equal_var=False)
                pooled_sd = np.sqrt(((len(comm_res)-1)*comm_res.std()**2 +
                                     (len(uncomm_res)-1)*uncomm_res.std()**2) /
                                    (len(comm_res)+len(uncomm_res)-2))
                d_eff = (comm_res.mean() - uncomm_res.mean()) / pooled_sd if pooled_sd > 0 else 0
                print(f"\n  Committed vs Uncommitted residual:")
                print(f"    Committed:   M={comm_res.mean():+.3f} (SD={comm_res.std():.3f}, n={len(comm_res)})")
                print(f"    Uncommitted: M={uncomm_res.mean():+.3f} (SD={uncomm_res.std():.3f}, n={len(uncomm_res)})")
                print(f"    t={t:.3f}, p={p:.4e}, d={d_eff:.3f}")

        # Reg 4: DV = persisted (logistic, committed only, SECONDARY)
        committed_df = df_m[df_m['committed_d1'] == 1].copy()
        if 'persisted_d2' in committed_df.columns and len(committed_df) > 15:
            print(f"\n  --- Reg 4 [SECONDARY]: DV = persisted (logistic, committed only, N={len(committed_df)}) ---")
            try:
                m4 = smf.logit('persisted_d2 ~ distance + p_mode_baseline + d1_commitment_strength',
                                data=committed_df).fit(disp=0,
                                cov_type='cluster', cov_kwds={'groups': committed_df['qid']})
                print(f"  Pseudo R² = {m4.prsquared:.4f}")
                for var in ['distance', 'p_mode_baseline', 'd1_commitment_strength']:
                    or_val = np.exp(m4.params[var])
                    ci = m4.conf_int().loc[var]
                    or_ci = [np.exp(ci[0]), np.exp(ci[1])]
                    print(f"    {var:30s}: OR={or_val:.3f} [{or_ci[0]:.3f},{or_ci[1]:.3f}], "
                          f"p={m4.pvalues[var]:.4e}")
            except Exception as e:
                print(f"    Failed: {e}")

    except Exception as e:
        print(f"{model_name}: FAILED — {e}")

SyntaxError: expected 'except' or 'finally' block (3241757145.py, line 10)

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 6: Study 2A — Categorical framing, planned contrasts
# ══════════════════════════════════════════════════════════

print("="*70)
print("STUDY 2A — Categorical framing, planned contrasts, cluster-robust")
print("="*70)

for model_name in df_s2['model'].unique():
    df_m = df_s2[(df_s2['model'] == model_name) &
                  (df_s2['experiment'] == '2A') &
                  (df_s2['stage'] == 'commit')].copy()
    if len(df_m) < 20: continue
    n_dilemmas = df_m['qid'].nunique()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)}, {n_dilemmas} dilemmas)")
    print(f"{'━'*60}")

    # Rates with Wilson CIs
    print("\n  Commitment rates [95% Wilson CI]:")
    for f in ['memory', 'instruction', 'suggestion']:
        sub = df_m[df_m['framing'] == f]
        n = len(sub)
        k = int(sub['committed'].sum())
        rate = k / n * 100
        ci_lo, ci_hi = proportion_confint(k, n, method='wilson')
        print(f"    {f:15s}: {rate:.1f}% [{ci_lo*100:.1f}, {ci_hi*100:.1f}] (n={n})")

    # Planned contrasts
    print("\n  Planned contrasts (Bonferroni α=.017):")
    for f1, f2 in [('memory', 'suggestion'), ('memory', 'instruction'), ('instruction', 'suggestion')]:
        sub = df_m[df_m['framing'].isin([f1, f2])]
        ct = pd.crosstab(sub['framing'], sub['committed'])
        if ct.shape[1] >= 2:
            chi2, p, _, _ = chi2_contingency(ct)
            a = df_m[(df_m['framing']==f1) & (df_m['committed']==1)].shape[0]
            b = df_m[(df_m['framing']==f1) & (df_m['committed']==0)].shape[0]
            c = df_m[(df_m['framing']==f2) & (df_m['committed']==1)].shape[0]
            d = df_m[(df_m['framing']==f2) & (df_m['committed']==0)].shape[0]
            or_val = (a*d)/(b*c) if b*c > 0 else float('inf')
            sig = '***' if p < .001 else ('**' if p < .017 else 'ns')
            print(f"    {f1} vs {f2}: χ²={chi2:.2f}, p={p:.4e}, OR={or_val:.2f} {sig}")

    # Logistic with categorical framing
    print("\n  Logistic (ref=suggestion, cluster-robust):")
    df_m['framing_cat'] = pd.Categorical(df_m['framing'],
                                          categories=['suggestion', 'instruction', 'memory'])
    try:
        logit_m = smf.logit('committed ~ C(framing_cat) + distance + p_mode_baseline',
                             data=df_m).fit(disp=0,
                             cov_type='cluster', cov_kwds={'groups': df_m['qid']})
        print(f"  Pseudo R² = {logit_m.prsquared:.4f}")
        for var in logit_m.params.index:
            if var == 'Intercept': continue
            or_val = np.exp(logit_m.params[var])
            ci = logit_m.conf_int().loc[var]
            or_ci = [np.exp(ci[0]), np.exp(ci[1])]
            print(f"    {var:50s}: OR={or_val:.3f} [{or_ci[0]:.3f},{or_ci[1]:.3f}], p={logit_m.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Logistic failed: {e}")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 7: Study 2B — Categorical attribution, planned contrasts
# ══════════════════════════════════════════════════════════

print("="*70)
print("STUDY 2B — Categorical attribution, planned contrasts, cluster-robust")
print("="*70)

for model_name in df_s2['model'].unique():
    df_m = df_s2[(df_s2['model'] == model_name) &
                  (df_s2['experiment'] == '2B') &
                  (df_s2['stage'] == 'commit')].copy()
    if len(df_m) < 20: continue
    n_dilemmas = df_m['qid'].nunique()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)}, {n_dilemmas} dilemmas)")
    print(f"{'━'*60}")

    # Rates
    print("\n  Commitment rates [95% Wilson CI]:")
    for a in ['self', 'user', 'other_ai']:
        sub = df_m[df_m['attribution'] == a]
        n = len(sub)
        k = int(sub['committed'].sum())
        rate = k / n * 100
        ci_lo, ci_hi = proportion_confint(k, n, method='wilson')
        print(f"    {a:15s}: {rate:.1f}% [{ci_lo*100:.1f}, {ci_hi*100:.1f}] (n={n})")

    # Planned contrasts
    print("\n  Planned contrasts (Bonferroni α=.017):")
    for a1, a2 in [('self', 'user'), ('self', 'other_ai'), ('user', 'other_ai')]:
        sub = df_m[df_m['attribution'].isin([a1, a2])]
        ct = pd.crosstab(sub['attribution'], sub['committed'])
        if ct.shape[1] >= 2:
            chi2, p, _, _ = chi2_contingency(ct)
            ac = df_m[(df_m['attribution']==a1) & (df_m['committed']==1)].shape[0]
            bc = df_m[(df_m['attribution']==a1) & (df_m['committed']==0)].shape[0]
            cc = df_m[(df_m['attribution']==a2) & (df_m['committed']==1)].shape[0]
            dc = df_m[(df_m['attribution']==a2) & (df_m['committed']==0)].shape[0]
            or_val = (ac*dc)/(bc*cc) if bc*cc > 0 else float('inf')
            sig = '***' if p < .001 else ('**' if p < .017 else 'ns')
            r1 = df_m[df_m['attribution']==a1]['committed'].mean()*100
            r2 = df_m[df_m['attribution']==a2]['committed'].mean()*100
            print(f"    {a1} ({r1:.1f}%) vs {a2} ({r2:.1f}%): χ²={chi2:.2f}, p={p:.4e}, OR={or_val:.2f} {sig}")
        else:
            # Complete separation — use Fisher exact
            ct_full = pd.crosstab(sub['attribution'], sub['committed'])
            if ct_full.shape == (2, 2):
                odds, p = stats.fisher_exact(ct_full)
                print(f"    {a1} vs {a2}: Fisher exact OR={odds:.3f}, p={p:.4e}")
            else:
                print(f"    {a1} vs {a2}: cannot compute (no variation)")

    # Logistic
    print("\n  Logistic (ref=other_ai, cluster-robust):")
    df_m['attr_cat'] = pd.Categorical(df_m['attribution'],
                                       categories=['other_ai', 'user', 'self'])
    try:
        logit_m = smf.logit('committed ~ C(attr_cat) + distance + p_mode_baseline',
                             data=df_m).fit(disp=0,
                             cov_type='cluster', cov_kwds={'groups': df_m['qid']})
        print(f"  Pseudo R² = {logit_m.prsquared:.4f}")
        for var in logit_m.params.index:
            if var == 'Intercept': continue
            or_val = np.exp(logit_m.params[var])
            ci = logit_m.conf_int().loc[var]
            or_ci = [np.exp(ci[0]), np.exp(ci[1])]
            print(f"    {var:50s}: OR={or_val:.3f} [{or_ci[0]:.3f},{or_ci[1]:.3f}], p={logit_m.pvalues[var]:.4e}")
    except Exception as e:
        print(f"    Logistic failed (likely separation): {e}")
        print("    Using Fisher exact for pairwise instead (reported above).")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 8: Study 2 — Persistence (secondary, conditional)
# ══════════════════════════════════════════════════════════

print("="*70)
print("STUDY 2 — PERSISTENCE (secondary: conditional on committed)")
print("="*70)

for exp, groupvar in [('2A', 'framing'), ('2B', 'attribution')]:
    print(f"\n{'━'*40} Experiment {exp} {'━'*40}")
    levels = ['memory','instruction','suggestion'] if exp == '2A' else ['self','user','other_ai']

    for model_name in df_s2['model'].unique():
        df_m = df_s2[(df_s2['model'] == model_name) & (df_s2['experiment'] == exp)]
        commit_df = df_m[df_m['stage'] == 'commit']
        correct_df = df_m[df_m['stage'] == 'correct']
        if len(commit_df) < 10: continue

        print(f"\n  {model_name}:")
        persist_data = []
        for lev in levels:
            committed = commit_df[(commit_df[groupvar] == lev) & (commit_df['committed'] == 1)]
            keys = set(zip(committed['qid'], committed['irr']))
            corrections = correct_df[
                (correct_df[groupvar] == lev) &
                correct_df.apply(lambda x: (x['qid'], x['irr']) in keys, axis=1)
            ]
            n_c = len(committed)
            n_p = corrections['committed'].sum() if len(corrections) > 0 else 0
            rate = n_p / n_c * 100 if n_c > 0 else 0
            ci_lo, ci_hi = proportion_confint(int(n_p), n_c, method='wilson') if n_c > 0 else (0, 0)
            print(f"    {lev:15s}: {rate:.1f}% [{ci_lo*100:.1f},{ci_hi*100:.1f}] (n={n_c})")
            for _, row in corrections.iterrows():
                persist_data.append({groupvar: lev, 'persisted': row['committed']})

        pdf = pd.DataFrame(persist_data)
        if len(pdf) > 0:
            ct = pd.crosstab(pdf[groupvar], pdf['persisted'])
            if ct.shape[1] >= 2:
                chi2, p, dof, _ = chi2_contingency(ct)
                print(f"    χ²({dof})={chi2:.3f}, p={p:.4f}")
            else:
                print(f"    Cannot test (no variation in persistence)")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 9: Study 3 — Categorical ratio + interaction, cluster-robust
# ══════════════════════════════════════════════════════════
# NOTE: This cell assumes df_s3 already exists from your Study 3 analysis notebook.
# If not, load it here:
# df_s3 = pd.read_csv("path/to/study3_df.csv")
df_s3 = 
print("="*70)
print("STUDY 3 — Categorical ratio + distance interaction, cluster-robust")
print("="*70)

for model_name in df_s3['model'].unique():
    df_m = df_s3[df_s3['model'] == model_name].copy()
    df_m = df_m.dropna(subset=['delta_p_initial', 'p_mode_baseline', 'base_extremity'])
    n_dilemmas = df_m['qid'].nunique()

    print(f"\n{'━'*60}")
    print(f"  {model_name} (N={len(df_m)}, {n_dilemmas} dilemmas)")
    print(f"{'━'*60}")

    df_m['ratio_cat'] = pd.Categorical(df_m['ratio'], categories=['0:3','1:2','2:1','3:0'])

    # Primary: categorical ratio
    print("\n  --- ΔP(initial) ~ ratio (categorical) + controls ---")
    try:
        m = smf.ols('delta_p_initial ~ C(ratio_cat) + p_mode_baseline + base_extremity',
                      data=df_m).fit(cov_type='cluster', cov_kwds={'groups': df_m['qid']})
        print(f"  R² = {m.rsquared:.4f}")
        for var in m.params.index:
            if var == 'Intercept': continue
            ci = m.conf_int().loc[var]
            print(f"    {var:45s}: β={m.params[var]:+.4f}, t={m.tvalues[var]:+.2f}, "
                  f"p={m.pvalues[var]:.4e}, CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")
    except Exception as e:
        print(f"    Failed: {e}")

    # Interaction: n_support × avg_dist_peers
    if 'avg_dist_peers' in df_m.columns:
        print("\n  --- + n_support × avg_dist_peers interaction ---")
        try:
            m2 = smf.ols('delta_p_initial ~ n_support * avg_dist_peers + p_mode_baseline + base_extremity',
                           data=df_m).fit(cov_type='cluster', cov_kwds={'groups': df_m['qid']})
            print(f"  R² = {m2.rsquared:.4f}")
            for var in m2.params.index:
                if var == 'Intercept': continue
                ci = m2.conf_int().loc[var]
                print(f"    {var:45s}: β={m2.params[var]:+.4f}, t={m2.tvalues[var]:+.2f}, "
                      f"p={m2.pvalues[var]:.4e}, CI=[{ci[0]:+.4f},{ci[1]:+.4f}]")
        except Exception as e:
            print(f"    Failed: {e}")

print("\n\nAll corrected analyses complete.")